# AI-Based Screening for Undiagnosed Sleep Apnea from a Single ECG Lead

### A two-stage deep learning study on the PhysioNet Apnea-ECG cohort

---

## The clinical problem

Obstructive sleep apnea affects a large fraction of adults, and the majority of cases are never
diagnosed. Left untreated it raises the risk of hypertension, stroke, atrial fibrillation, type 2
diabetes, and drowsy-driving collisions. It is also a well-established perioperative risk factor —
patients with undiagnosed OSA face elevated rates of postoperative respiratory complications.

The reason it stays undiagnosed is not that the disease is subtle. It is that the diagnostic test is
expensive. Polysomnography requires an overnight stay in a sleep laboratory, costs thousands of
dollars, and carries waiting lists measured in months.

Meanwhile, a **single ECG lead** is already being recorded almost everywhere: hospital telemetry,
Holter monitors, chest straps, smartwatches, and consumer patches.

> **Can an AI model read one ECG channel overnight and identify who needs a sleep study?**

That is a patient-safety question as much as a machine-learning one: the goal is not to replace
polysomnography, but to decide who should be sent for it.

---

## Two stages

| | Input | Output | Unit |
|---|---|---|---|
| **Stage 1** | One minute of single-lead ECG | Apnea present / absent | 1-minute segment (~34,000) |
| **Stage 2** | The whole night's predicted minutes | Clinically significant apnea, plus an estimated apnea index | Recording (n = 70) |

The reference standard is expert annotation: human scorers labelled each minute of each recording
using simultaneously recorded respiration signals. Those respiration signals are **never** given to
the model. Only the ECG is.

---

## A secondary question about the AI itself

Apneic minutes cluster in time — apnea comes in runs, not isolated minutes. So we test directly
whether letting the model see *neighbouring* minutes improves detection.

We do this with a **matched pair**: a 1D-CNN that sees one minute in isolation, and a CNN-BiLSTM
that wraps a **byte-identical convolutional encoder** and adds a recurrent layer across a window of
consecutive minutes. Same encoder, same folds, same hyperparameters. The performance gap therefore
isolates the value of temporal context, which the systematic review's Table 8 never tests directly.

---

## How this maps onto the systematic review

**Data type (RQ2).** Single-lead ECG, the second most common modality in the reviewed corpus.

**Preprocessing (RQ3).** Butterworth bandpass filtering, Pan-Tompkins R-peak detection, RR-interval
series construction, ECG-derived respiration, resampling to a uniform grid, per-recording
normalization, one-minute segmentation, and HRV feature extraction across time, frequency and
nonlinear domains.

**Models (RQ4).** Random Forest (Table 7), 1D-CNN and CNN-BiLSTM (Table 8).

Because Apnea-ECG is the benchmark most of the reviewed studies used, the results table below is
directly comparable to theirs.

---

## Running this on Kaggle

**No internet required.** Everything reads from disk.

1. Download `apnea-ecg-database-1.0.0.zip` from PhysioNet (Open Access — no login).
2. Kaggle → **Datasets → New Dataset** → upload the ZIP → name it `apnea-ecg`.
3. In this notebook → **Add Input** → Datasets → select it.
4. Settings → Accelerator → **GPU**.

The notebook searches recursively for the records, so the folder depth produced by extraction does
not matter. There is also **no `wfdb` dependency**: the WFDB header, signal and annotation readers
are implemented directly in section 1, because `pip install` needs internet that this notebook does
not assume.

---
# 0. Configuration

In [ ]:
import os, re, gc, io, time, json, glob, struct, warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, filtfilt, welch, find_peaks, resample_poly
from scipy.interpolate import PchipInterpolator
from scipy.stats import iqr

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             confusion_matrix, roc_auc_score, roc_curve,
                             recall_score, precision_score,
                             balanced_accuracy_score, mean_absolute_error)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("numpy", np.__version__, "| pandas", pd.__version__, "| torch", torch.__version__)

In [ ]:
@dataclass
class Config:
    # ---------------- Paths -------------------------------------------------
    # Searched recursively, so extraction depth does not matter.
    INPUT_ROOT: str = "/kaggle/input"
    CACHE_DIR: str = "/kaggle/working/cache"
    OUT_DIR: str = "/kaggle/working/outputs"

    # ---------------- Signal ------------------------------------------------
    FS: int = 100                 # Apnea-ECG native sampling rate
    SEGMENT_SEC: int = 60         # annotations are one per minute
    ECG_BAND: tuple = (0.5, 45.0) # Hz, Butterworth bandpass on raw ECG
    BUTTER_ORDER: int = 3         # "third-order Butterworth" per the review

    # Derived series (RRI and EDR) are resampled onto this uniform grid.
    # 4 Hz is the HRV convention: well above twice the 0.4 Hz upper edge of the
    # HF band, so nothing of interest is aliased, and it keeps the tensor small.
    DERIVED_FS: float = 4.0

    # ---------------- Modelling ---------------------------------------------
    CONTEXT: int = 5              # +/- 5 minutes -> 11-minute window for BiLSTM
    N_FOLDS: int = 5
    BATCH_SIZE: int = 128
    EPOCHS: int = 30
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    PATIENCE: int = 6

    # ---------------- Stage 2 -----------------------------------------------
    # The challenge's own criterion for a Class A (apneic) recording: at least
    # 100 minutes of apnea across the night. We use the dataset's definition
    # rather than inventing one.
    APNEA_MINUTES_THRESHOLD: int = 100

    # ---------------- Debug --------------------------------------------------
    LIMIT_RECORDS: int = None     # set to e.g. 6 for a fast smoke test

CFG = Config()
CFG.SEG_LEN = CFG.FS * CFG.SEGMENT_SEC          # 6000 raw ECG samples per minute
CFG.DERIVED_LEN = int(CFG.DERIVED_FS * CFG.SEGMENT_SEC)   # 240 samples per minute
CFG.WINDOW = 2 * CFG.CONTEXT + 1                # 11 minutes of context

for d in (CFG.CACHE_DIR, CFG.OUT_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

def select_device():
    """
    Pick a device, and VERIFY it actually works before we rely on it.

    Why the smoke test: `torch.cuda.is_available()` returning True is not
    enough. It only says a CUDA driver and device are present, not that this
    PyTorch build has compiled kernels for that device's compute capability.
    Kaggle's Tesla P100 is compute capability 6.0 (Pascal), and PyTorch builds
    against CUDA 13 dropped Pascal support -- so is_available() says True and
    then the first convolution raises

        CUDA error: no kernel image is available for execution on the device

    twenty minutes into training. Running one tiny conv here surfaces that in
    two seconds instead, and falls back to CPU so the notebook still completes.
    """
    if not torch.cuda.is_available():
        print("No CUDA device. Settings -> Accelerator -> GPU (T4 x2).")
        return torch.device("cpu")

    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    arch_list = torch.cuda.get_arch_list()
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  (compute capability {cap[0]}.{cap[1]}, {mem:.1f} GB)")
    print(f"PyTorch {torch.__version__} compiled for: {arch_list}")

    try:
        # A real convolution on the device, matching what the models do.
        probe = nn.Conv1d(2, 4, 3).cuda()
        out = probe(torch.randn(2, 2, 32, device="cuda"))
        torch.cuda.synchronize()
        del probe, out
        torch.cuda.empty_cache()
        print("CUDA smoke test passed.")
        return torch.device("cuda")
    except Exception as ex:
        print(f"\n*** CUDA IS PRESENT BUT UNUSABLE ***")
        print(f"    {type(ex).__name__}: {str(ex).splitlines()[0]}")
        if f"sm_{cap[0]}{cap[1]}" not in " ".join(arch_list):
            print(f"\n    This PyTorch build has no kernels for sm_{cap[0]}{cap[1]}.")
            print(f"    FIX: Settings -> Accelerator -> GPU T4 x2, then re-run.")
            print(f"    (T4 is compute capability 7.5 and is supported.)")
        print(f"\n    Falling back to CPU. Training will be much slower --")
        print(f"    switch accelerator rather than sitting through it.\n")
        return torch.device("cpu")


DEVICE = select_device()
print(f"\nUsing device: {DEVICE}")

print(f"\nSegment = {CFG.SEGMENT_SEC}s @ {CFG.FS}Hz = {CFG.SEG_LEN} raw samples")
print(f"Derived series = {CFG.DERIVED_LEN} samples/min @ {CFG.DERIVED_FS}Hz")
print(f"Context window = {CFG.WINDOW} minutes")

---
# 1. Reading WFDB files without the `wfdb` package

Apnea-ECG is distributed in PhysioNet's WFDB format: a plain-text `.hea` header, a binary `.dat`
signal file, and a binary `.apn` annotation file. The usual way to read these is the `wfdb` Python
package, but installing it needs internet, which this notebook does not assume. The format is
simple enough to read directly.

**`.hea`** — first line gives record name, number of signals, sampling frequency, and sample count.
Each following line describes one signal: filename, storage format, gain and baseline, ADC
resolution and zero, and a description. Physical units are recovered as `(raw - baseline) / gain`.

**`.dat`** — Apnea-ECG uses **format 16**: signed 16-bit little-endian integers, one per sample. We
also handle **format 212** (two 12-bit samples packed into three bytes) so the same reader works on
other PhysioNet databases such as UCDDB if you add one later.

**`.apn`** — a stream of little-endian 16-bit words. The top 6 bits are a type code and the bottom
10 bits are the time elapsed since the previous annotation. Because one minute at 100 Hz is 6000
samples, and 6000 does not fit in 10 bits, the file uses **SKIP** records (code 59) carrying a
32-bit interval in the following two words. Getting this detail wrong silently produces annotations
at the wrong times, so it is handled explicitly.

For Apnea-ECG the symbols are `N` (no apnea during this minute) and `A` (apnea).

In [ ]:
# ===========================================================================
# 1.a  WFDB readers
# ===========================================================================

WFDB_ANN_SYMBOLS = {
    0: "", 1: "N", 2: "L", 3: "R", 4: "a", 5: "V", 6: "F", 7: "J", 8: "A",
    9: "S", 10: "E", 11: "j", 12: "/", 13: "Q", 14: "~", 16: "|", 18: "s",
    19: "T", 20: "*", 21: "D", 22: '"', 23: "=", 24: "p", 25: "B", 26: "^",
    27: "t", 28: "+", 29: "u", 30: "?", 31: "!", 32: "[", 33: "]", 34: "e",
    35: "n", 36: "@", 37: "x", 38: "f", 39: "(", 40: ")", 41: "r",
}
ANN_SKIP, ANN_NUM, ANN_SUB, ANN_CHN, ANN_AUX = 59, 60, 61, 62, 63


def read_header(base):
    """Parse a WFDB .hea header. `base` is the path without extension."""
    with open(base + ".hea", "r") as fh:
        lines = [l.strip() for l in fh if l.strip() and not l.startswith("#")]

    p0 = lines[0].split()
    hdr = dict(record_name=p0[0], n_sig=int(p0[1]),
               fs=float(p0[2]) if len(p0) > 2 else 250.0,
               n_samples=int(p0[3]) if len(p0) > 3 else 0, signals=[])

    for line in lines[1:1 + hdr["n_sig"]]:
        p = line.split()
        fmt = int(re.match(r"(\d+)", p[1]).group(1))

        # Gain "200(0)/mV" -> gain 200, baseline 0. A gain of 0 means
        # uncalibrated, for which WFDB specifies a default of 200.
        gain, baseline = 200.0, 0
        if len(p) > 2:
            m = re.match(r"([-\d.]+)(?:\((-?\d+)\))?", p[2])
            if m:
                gain = float(m.group(1)) or 200.0
                if m.group(2) is not None:
                    baseline = int(m.group(2))
        adc_zero = int(p[4]) if len(p) > 4 else 0
        if len(p) > 2 and "(" not in p[2]:
            baseline = adc_zero

        hdr["signals"].append(dict(
            file_name=p[0], fmt=fmt, gain=gain, baseline=baseline,
            adc_zero=adc_zero,
            desc=" ".join(p[8:]) if len(p) > 8 else f"sig{len(hdr['signals'])}"))
    return hdr


def read_signal(base, hdr=None):
    """Read a WFDB .dat file and return physical units, shape (n_samples, n_sig)."""
    hdr = hdr or read_header(base)
    dat = os.path.join(os.path.dirname(base), hdr["signals"][0]["file_name"])
    fmt, nsig = hdr["signals"][0]["fmt"], hdr["n_sig"]

    if fmt == 16:
        raw = np.fromfile(dat, dtype="<i2").astype(np.int32)
    elif fmt == 212:
        b = np.fromfile(dat, dtype=np.uint8)
        n3 = len(b) // 3
        b = b[:n3 * 3].reshape(-1, 3).astype(np.int32)
        s1 = b[:, 0] | ((b[:, 1] & 0x0F) << 8)
        s2 = b[:, 2] | ((b[:, 1] >> 4) << 8)
        s1 = np.where(s1 > 2047, s1 - 4096, s1)      # sign-extend 12 bits
        s2 = np.where(s2 > 2047, s2 - 4096, s2)
        raw = np.empty(n3 * 2, dtype=np.int32)
        raw[0::2], raw[1::2] = s1, s2
    else:
        raise NotImplementedError(f"WFDB format {fmt} not supported")

    raw = (raw[:len(raw) // nsig * nsig].reshape(-1, nsig) if nsig > 1
           else raw.reshape(-1, 1))
    out = np.empty(raw.shape, dtype=np.float64)
    for i, s in enumerate(hdr["signals"][:nsig]):
        out[:, i] = (raw[:, i] - s["baseline"]) / s["gain"]
    return out, hdr


def read_annotations(base, ext="apn"):
    """
    Read a WFDB binary annotation file -> (sample_indices, symbols).

    Word layout: top 6 bits = type code, bottom 10 bits = interval since the
    previous annotation. Because a one-minute interval at 100 Hz is 6000
    samples and will not fit in 10 bits, .apn files use SKIP (code 59) with a
    32-bit interval in the next two words, high word first. Mishandling that
    would place every annotation at the wrong time, so it is explicit here.
    """
    with open(base + "." + ext, "rb") as fh:
        data = np.frombuffer(fh.read(), dtype="<u2")

    samples, symbols, t, i = [], [], 0, 0
    while i < len(data):
        word = int(data[i]); code = word >> 10; interval = word & 0x03FF

        if code == 0 and interval == 0:
            break                                   # end of file
        if code == ANN_SKIP:
            if i + 2 >= len(data):
                break
            t += (int(data[i + 1]) << 16) | int(data[i + 2])
            i += 3
            continue
        if code in (ANN_NUM, ANN_SUB, ANN_CHN):
            i += 2
            continue
        if code == ANN_AUX:
            i += 1 + (interval + 1) // 2            # padded to a word boundary
            continue

        t += interval
        samples.append(t)
        symbols.append(WFDB_ANN_SYMBOLS.get(code, "?"))
        i += 1

    return np.array(samples, dtype=np.int64), symbols


print("WFDB readers defined (no external dependency).")

In [ ]:
# ===========================================================================
# 1.b  Locate the records
# ===========================================================================

def find_records(root):
    """
    Find every record that has BOTH a .hea and a .apn file.

    Searched recursively across ALL attached datasets. Three things this has to
    handle that a naive search does not:

      * The SAME record can appear in several attached datasets. We keep the
        copy that has annotations beside it. Taking whichever .hea appears
        first would silently drop a record whose labels live elsewhere -- which
        is exactly how the 35 test records went missing on the first attempt.

      * Companion files. Records named a01r / a01er carry the extra respiration
        signals and are the SAME PATIENTS as a01-a04. Including them would put
        one person on both sides of a split. The strict record-name pattern
        excludes them.

      * Some mirrors ship a challenge/ subfolder of duplicates, skipped by path.
    """
    candidates = {}
    for h in sorted(glob.glob(os.path.join(root, "**", "*.hea"), recursive=True)):
        base = h[:-4]
        name = os.path.basename(base)
        if not re.fullmatch(r"[abcx]\d{2}", name):
            continue
        if os.sep + "challenge" + os.sep in h:
            continue
        candidates.setdefault(name, []).append(base)

    recs, no_ann = {}, []
    for name, bases in candidates.items():
        with_ann = [b for b in bases if os.path.exists(b + ".apn")]
        if with_ann:
            recs[name] = with_ann[0]
        else:
            no_ann.append(name)
    return recs, sorted(no_ann)


RECORDS, MISSING_ANN = find_records(CFG.INPUT_ROOT)
print(f"Found {len(RECORDS)} annotated records under {CFG.INPUT_ROOT}")

if not RECORDS:
    raise RuntimeError(
        "No records found.\n"
        "  * Add Input -> Datasets -> your apnea-ecg dataset\n"
        f"  * Present: {sorted(os.listdir(CFG.INPUT_ROOT))}")

names = sorted(RECORDS)
n_learn = sum(1 for n in names if n[0] in "abc")
n_test = sum(1 for n in names if n[0] == "x")
print(f"  learning set (a/b/c): {n_learn}")
print(f"  test set (x)        : {n_test}")

if MISSING_ANN:
    print(f"\n  {len(MISSING_ANN)} records have signals but no annotations and "
          f"will be skipped: {MISSING_ANN[:8]}{' ...' if len(MISSING_ANN) > 8 else ''}")

if n_test == 0:
    print("\n  No test records -- the notebook runs on cross-validation only.")
    print("  To enable the official held-out evaluation, attach a dataset that")
    print("  includes x01.apn ... x35.apn (they are absent from some mirrors).")
else:
    print(f"\n  Official held-out test set available ({n_test} records).")

print(f"\n  a/b/c from: {os.path.dirname(RECORDS[names[0]])}")
if n_test:
    _fx = next(n for n in names if n[0] == "x")
    print(f"  x     from: {os.path.dirname(RECORDS[_fx])}")

In [ ]:
# ===========================================================================
# 1.c  Read every record's annotations -> the cohort table
# ===========================================================================
# Reading only the annotations first (they are tiny) lets us describe the
# cohort and check class balance BEFORE spending time on signal processing.
#
# Ground truth per recording is derived from the annotations themselves rather
# than parsed out of additional-information.txt, because the annotations are
# unambiguous and are present for all 70 records including the test set.

rows = []
for name in sorted(RECORDS):
    base = RECORDS[name]
    try:
        samp, sym = read_annotations(base, "apn")
        hdr = read_header(base)
        n_min = len(sym)
        n_apnea = sum(1 for s in sym if s == "A")
        rows.append(dict(
            record=name,
            group="learning" if name[0] in "abc" else "test",
            letter=name[0],
            fs=hdr["fs"],
            hours=hdr["n_samples"] / hdr["fs"] / 3600 if hdr["n_samples"] else np.nan,
            n_minutes=n_min,
            apnea_minutes=n_apnea,
            apnea_frac=n_apnea / max(n_min, 1),
            # Apnea index: apneic minutes per hour of recording. This is not the
            # clinical AHI (which counts individual events), but it is the
            # per-recording severity measure this dataset supports and it
            # correlates strongly with AHI.
            apnea_index=60.0 * n_apnea / max(n_min, 1),
        ))
    except Exception as ex:
        print(f"  {name}: FAILED {type(ex).__name__}: {ex}")

COHORT = pd.DataFrame(rows).set_index("record")

# The dataset's own Class A criterion.
COHORT["apneic"] = (COHORT["apnea_minutes"] >= CFG.APNEA_MINUTES_THRESHOLD).astype(int)

print(f"Cohort: {len(COHORT)} recordings, "
      f"{COHORT['n_minutes'].sum():,} annotated minutes "
      f"({COHORT['hours'].sum():.0f} hours)")
print(f"\nApneic minutes overall: {COHORT['apnea_minutes'].sum():,} "
      f"({100*COHORT['apnea_minutes'].sum()/COHORT['n_minutes'].sum():.1f}%)")
print(f"Apneic recordings (>= {CFG.APNEA_MINUTES_THRESHOLD} apnea min): "
      f"{COHORT['apneic'].sum()} / {len(COHORT)}")

print("\nBy record letter (a = apnea, b = borderline, c = control, x = test):")
print(COHORT.groupby("letter").agg(
    n=("n_minutes", "size"),
    minutes=("n_minutes", "sum"),
    apnea_min=("apnea_minutes", "sum"),
    mean_apnea_index=("apnea_index", "mean")).round(1).to_string())

display(COHORT.head(10).round(2))

In [ ]:
# ===========================================================================
# 1.d  Cohort figure
# ===========================================================================
# Three panels that establish the framing before any modelling:
#   (a) apnea burden varies enormously across recordings -- some nights are
#       almost entirely apneic, others almost entirely clear.
#   (b) the per-minute class balance, which is the number every Stage 1
#       accuracy must be read against.
#   (c) recording durations, confirming these are full nights.

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].hist(COHORT["apnea_index"], bins=25, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("Apneic minutes per hour")
axes[0].set_ylabel("Recordings")
axes[0].set_title("(a) Apnea burden across the cohort")

frac = COHORT["apnea_minutes"].sum() / COHORT["n_minutes"].sum()
axes[1].bar(["No apnea", "Apnea"], [1 - frac, frac],
            color=["#55A868", "#C44E52"])
axes[1].set_ylabel("Fraction of all minutes")
axes[1].set_title(f"(b) Per-minute class balance\n(apnea = {100*frac:.1f}%)")
for i, v in enumerate([1 - frac, frac]):
    axes[1].text(i, v + 0.01, f"{100*v:.1f}%", ha="center")

axes[2].hist(COHORT["hours"].dropna(), bins=20, color="#8172B2", edgecolor="white")
axes[2].set_xlabel("Recording duration (hours)")
axes[2].set_ylabel("Recordings")
axes[2].set_title("(c) Night length")

plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig1_cohort.png", dpi=150, bbox_inches="tight")
plt.show()

---
# 2. Preprocessing

Every step below is one of the review's RQ3 categories, applied in a deliberate order.

### 2.1 Why apnea is visible in an ECG at all

This is the physiological premise of the whole study, so it is worth stating plainly.

During an obstructive apnea the airway collapses while the patient keeps trying to breathe. Three
things follow, and all three leave a mark on the ECG:

- **Cyclical variation in heart rate.** Apnea produces bradycardia during the obstruction followed
  by an abrupt tachycardia at the arousal that ends it. This slow oscillation, on a roughly
  30–60 second cycle, is the single strongest ECG signature of apnea and is why heart-rate
  variability features dominate this task.
- **Sympathetic surges.** Each arousal is accompanied by an autonomic shift that alters the balance
  between low- and high-frequency HRV power.
- **Mechanical modulation of the R-wave.** Chest movement and changing thoracic impedance during
  respiratory effort modulate the *amplitude* of the R peak. Tracking that amplitude recovers a
  respiration signal from the ECG alone — the **ECG-derived respiration** (EDR) — without any
  respiratory sensor.

So we extract two derived series from the ECG: the **RR-interval** series (rhythm) and the **EDR**
series (breathing). Together they carry most of what a respiratory belt would have told us.

### 2.2 Why a third-order Butterworth bandpass at 0.5–45 Hz

**Butterworth** because it has a maximally flat passband. Chebyshev and elliptic designs give
sharper roll-off but introduce passband ripple, which distorts QRS *morphology* — exactly what the
CNN reads and what the R-peak detector depends on.

**Third order**, applied with `filtfilt` (forwards then backwards) for **zero phase shift**. This is
not cosmetic. We measure R-peak *timing* to build the RR-interval series, and a filter that shifted
peaks by even 10 ms would bias every HRV feature systematically.

**0.5 Hz high-pass** removes baseline wander from respiration and electrode motion. **45 Hz
low-pass** removes muscle noise and sits below both 50 Hz and 60 Hz mains interference, so it
doubles as a mains filter without needing a separate notch — convenient here because Apnea-ECG was
recorded in Europe (50 Hz) while much published code assumes 60 Hz.

### 2.3 Why Pan-Tompkins for R-peak detection

The review names Pan-Tompkins as the standard ECG R-peak detector, and its design is purpose-built
for this signal. The pipeline is: bandpass to isolate QRS energy → differentiate to emphasise the
steep QRS slope → square to make everything positive and amplify large deflections → integrate over
a moving window roughly the width of a QRS complex → threshold adaptively.

The **adaptive** threshold is the important part. ECG amplitude drifts over a night with posture and
electrode contact, and a fixed threshold would progressively miss beats as the night went on.

### 2.4 Why the RR series must be resampled

The RR-interval series is inherently **unevenly sampled** — one value per heartbeat, and the
heartbeats are not evenly spaced. That is a problem, because Welch's method for spectral estimation
assumes uniform sampling. So we interpolate onto a uniform 4 Hz grid before any frequency-domain
analysis.

We use **PCHIP** rather than a cubic spline because it is shape-preserving and cannot overshoot
between beats. A spline can ring around an abrupt rate change — precisely what an apnea termination
produces — and manufacture RR intervals that were never measured.

### 2.5 Why normalize per recording

ECG amplitude depends on electrode placement, skin impedance, and body habitus, all constant within
a recording and variable between patients. Normalizing globally would let the model identify *which
patient* a segment came from, and patient identity correlates with label. Normalizing within each
recording removes that shortcut, and it is deployable: a real overnight monitor has the whole night.

We use a **robust** z-score, `(x − median) / IQR`, rather than mean and standard deviation. A single
motion artefact can be many times the physiological range, and it would drag a mean/SD
normalization far enough to compress the real signal into a narrow band near zero. Median and IQR
ignore it.

In [ ]:
# ===========================================================================
# 2.a  Filters
# ===========================================================================

def butter_bandpass(x, fs, lo, hi, order=3):
    """
    Zero-phase third-order Butterworth bandpass (review RQ3, "Filtering").

    filtfilt applies the filter forwards then backwards. This squares the
    magnitude response (effective order 2*order) but cancels phase entirely.
    Zero phase is essential here: we use R-peak TIMING to build the RR series,
    and any phase shift would bias every HRV feature in the same direction.
    """
    nyq = 0.5 * fs
    b, a = butter(order, [lo / nyq, min(hi / nyq, 0.99)], btype="band")
    if len(x) <= 3 * max(len(a), len(b)):
        return x
    return filtfilt(b, a, x)


def pchip_to_grid(times, values, grid):
    """
    Shape-preserving interpolation onto a uniform grid (review RQ3,
    "Interpolation").

    PCHIP is monotonicity-preserving, so it cannot overshoot between knots. A
    natural cubic spline can ring around the abrupt heart-rate change that ends
    an apnea and invent RR intervals outside the observed range. Outside the
    observed support we hold the nearest value rather than extrapolate, because
    extrapolating a cubic past its data is how you get a 20 ms RR interval.
    """
    ok = np.isfinite(values)
    if ok.sum() < 4:
        return np.full(len(grid), np.nan)
    out = PchipInterpolator(times[ok], values[ok], extrapolate=False)(grid)
    if np.isnan(out).any():
        out = pd.Series(out).ffill().bfill().to_numpy()
    return out


def robust_z(x):
    """Robust per-recording normalization; clipped to suppress residual spikes."""
    x = np.asarray(x, dtype=np.float64)
    finite = x[np.isfinite(x)]
    if finite.size == 0:
        return np.zeros_like(x)
    med = np.median(finite)
    scale = iqr(finite)
    scale = scale if scale > 1e-9 else (np.std(finite) or 1.0)
    return np.clip((x - med) / scale, -10, 10)


print("Filters defined.")

In [ ]:
# ===========================================================================
# 2.b  Pan-Tompkins R-peak detection  (review RQ3, "Peak Detection")
# ===========================================================================

def pan_tompkins(ecg, fs):
    """
    Detect R peaks using the Pan-Tompkins algorithm.

    Stage by stage, and why each stage exists:

      1. BANDPASS 5-15 Hz. QRS energy is concentrated here. This is a NARROWER
         band than the 0.5-45 Hz used for morphology, because for DETECTION we
         want to suppress P and T waves (mostly below 5 Hz) that would otherwise
         trigger false positives.

      2. DIFFERENTIATE. The QRS complex is defined by its steep slope, not its
         height. Differentiating turns "steep" into "large".

      3. SQUARE. Makes everything positive and amplifies large values
         disproportionately, widening the gap between QRS and noise.

      4. MOVING-WINDOW INTEGRATE over ~150 ms, about the width of a QRS. This
         merges the multiple peaks of one QRS complex into a single smooth bump,
         so each beat produces one detection rather than three.

      5. ADAPTIVE THRESHOLD. The threshold tracks the running signal level
         instead of being fixed. Over a 9-hour recording, ECG amplitude drifts
         with posture and electrode contact; a fixed threshold would steadily
         lose beats as the night progressed.

    A 200 ms refractory period enforces the physiological minimum between
    beats (300 bpm), preventing the T wave from being counted as a second R.

    Returns R-peak sample indices.
    """
    # 1. Bandpass for detection.
    x = butter_bandpass(ecg, fs, 5.0, min(15.0, 0.45 * fs), order=2)

    # 2. Differentiate (5-point derivative).
    d = np.convolve(x, np.array([1, 2, 0, -2, -1]) * (fs / 8.0), mode="same")

    # 3. Square.
    sq = d ** 2

    # 4. Moving-window integration (~150 ms).
    w = max(1, int(0.150 * fs))
    integrated = np.convolve(sq, np.ones(w) / w, mode="same")

    # 5. Adaptive thresholding.
    # Seed from the whole-signal statistics, then refine locally. A long
    # rolling median tracks slow amplitude drift without reacting to
    # individual beats.
    refractory = int(0.200 * fs)
    win = int(2.0 * fs)
    if len(integrated) > win:
        local = pd.Series(integrated).rolling(win, center=True,
                                              min_periods=1).median().to_numpy()
    else:
        local = np.full_like(integrated, np.median(integrated))
    thr = local + 0.5 * np.std(integrated)

    cand, _ = find_peaks(integrated, height=thr, distance=refractory)

    # Refine each detection back onto the true R peak in the morphology-filtered
    # signal: the integration window shifts peaks slightly, and we care about
    # exact timing.
    search = int(0.050 * fs)
    peaks = []
    for c in cand:
        lo, hi = max(0, c - search), min(len(ecg), c + search)
        if hi > lo:
            peaks.append(lo + int(np.argmax(np.abs(ecg[lo:hi]))))
    return np.unique(np.asarray(peaks, dtype=np.int64))


def rr_and_edr(ecg, peaks, fs):
    """
    Build the two derived series from detected R peaks.

    RR (rhythm): successive peak-to-peak intervals in seconds. Physiologically
    implausible intervals are removed -- outside 0.3-2.0 s (30-200 bpm), or more
    than 20% from the local median. This filtering is not optional. A single
    missed beat creates an interval near double the true one, and RMSSD is built
    from SUCCESSIVE DIFFERENCES, so it is maximally sensitive to exactly that
    error.

    EDR (breathing): the R-peak AMPLITUDE series. Respiratory effort moves the
    chest and changes thoracic impedance, modulating how tall the R wave looks.
    Tracking peak height therefore recovers a respiration surrogate from the ECG
    alone -- which is the point, since we are forbidden from using the actual
    respiration channels that produced the labels.

    Returns (rr_times, rr_values, edr_times, edr_values).
    """
    if len(peaks) < 4:
        return (np.array([]),) * 4

    t = peaks / fs
    rr = np.diff(t)
    rr_t = t[1:]

    keep = (rr >= 0.3) & (rr <= 2.0)
    rr, rr_t = rr[keep], rr_t[keep]
    if len(rr) < 4:
        return (np.array([]),) * 4

    med = pd.Series(rr).rolling(5, center=True, min_periods=1).median().to_numpy()
    keep2 = np.abs(rr - med) <= 0.20 * med
    rr, rr_t = rr[keep2], rr_t[keep2]

    edr_v = np.abs(ecg[peaks])
    edr_t = t
    return rr_t, rr, edr_t, edr_v


print("Pan-Tompkins defined.")

In [ ]:
# ===========================================================================
# 2.c  Per-minute features  (review RQ3, "Feature Extraction")
# ===========================================================================

def sample_entropy(x, m=2, r_factor=0.2):
    """
    Sample entropy: the negative log probability that two sequences matching
    for m points still match for m+1. Low means predictable, high means
    irregular.

    Relevant here because apnea imposes a CYCLICAL bradycardia-tachycardia
    pattern on the RR series. That changes its regularity in a way that mean and
    variance cannot express.
    """
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < m + 15:
        return np.nan
    r = r_factor * np.std(x)
    if r <= 0:
        return np.nan

    def count(mm):
        tpl = np.lib.stride_tricks.sliding_window_view(x, mm)
        d = np.abs(tpl[:, None, :] - tpl[None, :, :]).max(axis=2)
        np.fill_diagonal(d, np.inf)
        return int(np.sum(d <= r))

    try:
        B, A = count(m), count(m + 1)
    except (MemoryError, ValueError):
        return np.nan
    return -np.log(A / B) if (A > 0 and B > 0) else np.nan


def band_power(f, pxx, lo, hi):
    """
    Rectangular integration of a Welch PSD over a frequency band.

    NOTE: np.trapz was REMOVED in NumPy 2.0 (renamed np.trapezoid), and Kaggle
    images ship both major versions. Summing bins times the frequency resolution
    is equivalent for a uniformly spaced PSD and works on every version.
    """
    m = (f >= lo) & (f < hi)
    if not m.any():
        return 0.0
    return float(np.sum(pxx[m]) * (f[1] - f[0]))


def segment_features(rr_t, rr, edr_t, edr, t0, t1):
    """
    Features for the minute spanning [t0, t1).

    Three families, each answering a different question:

      RHYTHM (time domain) -- how much does heart rate vary overall (SDNN) and
        between consecutive beats (RMSSD, pNN50)? RMSSD and pNN50 reflect fast
        vagal changes, which drop sharply at an apnea-terminating arousal.

      RHYTHM (frequency domain) -- LF (0.04-0.15 Hz) mixes sympathetic and
        parasympathetic drive; HF (0.15-0.40 Hz) is respiratory sinus arrhythmia
        and is essentially parasympathetic. Apnea shifts the balance
        sympathetic-ward, so LF/HF should rise. We also measure a VLF-adjacent
        "apnea band" around 0.01-0.04 Hz, because the apnea cycle itself
        (roughly 30-60 s) lands there -- this is the most directly
        disease-specific spectral feature in the set.

      RESPIRATION (EDR) -- amplitude, variability and spectral peak of the
        ECG-derived respiration signal. During an obstructive event, effort
        continues while airflow stops, so EDR amplitude and regularity change
        characteristically.
    """
    out = {k: np.nan for k in [
        "rr_mean", "rr_sdnn", "rr_rmssd", "rr_pnn50", "rr_cv", "rr_range",
        "rr_min", "rr_max", "rr_n_beats", "rr_hr_mean",
        "lf", "hf", "lf_hf", "vlf_apnea", "total_power", "lf_norm", "hf_norm",
        "sd1", "sd2", "sd_ratio", "sampen",
        "edr_mean", "edr_std", "edr_cv", "edr_peak_freq", "edr_power",
        "edr_range"]}

    m = (rr_t >= t0) & (rr_t < t1)
    seg_rr = rr[m]
    out["rr_n_beats"] = float(len(seg_rr))

    if len(seg_rr) >= 5:
        out["rr_mean"] = float(np.mean(seg_rr))
        out["rr_hr_mean"] = float(60.0 / np.mean(seg_rr))
        out["rr_sdnn"] = float(np.std(seg_rr, ddof=1))
        out["rr_range"] = float(np.ptp(seg_rr))
        out["rr_min"], out["rr_max"] = float(np.min(seg_rr)), float(np.max(seg_rr))
        out["rr_cv"] = float(out["rr_sdnn"] / out["rr_mean"]) if out["rr_mean"] else np.nan
        d = np.diff(seg_rr)
        out["rr_rmssd"] = float(np.sqrt(np.mean(d ** 2)))
        out["rr_pnn50"] = float(np.mean(np.abs(d) > 0.050) * 100.0)

        # Poincare descriptors have closed forms in terms of SDNN and RMSSD.
        sd1 = out["rr_rmssd"] / np.sqrt(2)
        sd2_sq = 2 * out["rr_sdnn"] ** 2 - 0.5 * out["rr_rmssd"] ** 2
        sd2 = np.sqrt(sd2_sq) if sd2_sq > 0 else np.nan
        out["sd1"] = float(sd1)
        out["sd2"] = float(sd2) if sd2 == sd2 else np.nan
        out["sd_ratio"] = float(sd1 / sd2) if (sd2 == sd2 and sd2 > 0) else np.nan
        out["sampen"] = sample_entropy(seg_rr)

    # Frequency domain needs an evenly sampled series, hence the 4 Hz grid.
    # One minute is short for 0.04 Hz resolution, so we widen the window to
    # +/- 90 s around the minute for spectral features only.
    wide = (rr_t >= t0 - 90) & (rr_t < t1 + 90)
    if wide.sum() >= 20:
        try:
            grid = np.arange(rr_t[wide][0], rr_t[wide][-1], 1.0 / CFG.DERIVED_FS)
            if len(grid) > 32:
                series = pchip_to_grid(rr_t[wide], rr[wide], grid)
                series = series - np.mean(series)          # detrend the DC term
                f, pxx = welch(series, fs=CFG.DERIVED_FS,
                               nperseg=min(256, len(series)))
                out["lf"] = band_power(f, pxx, 0.04, 0.15)
                out["hf"] = band_power(f, pxx, 0.15, 0.40)
                out["vlf_apnea"] = band_power(f, pxx, 0.01, 0.04)
                out["total_power"] = band_power(f, pxx, 0.003, 0.40)
                out["lf_hf"] = out["lf"] / out["hf"] if out["hf"] > 0 else np.nan
                denom = out["lf"] + out["hf"]
                if denom > 0:
                    out["lf_norm"] = out["lf"] / denom
                    out["hf_norm"] = out["hf"] / denom
        except Exception:
            pass

    me = (edr_t >= t0) & (edr_t < t1)
    seg_edr = edr[me]
    if len(seg_edr) >= 5:
        out["edr_mean"] = float(np.mean(seg_edr))
        out["edr_std"] = float(np.std(seg_edr))
        out["edr_range"] = float(np.ptp(seg_edr))
        out["edr_cv"] = float(out["edr_std"] / out["edr_mean"]) if out["edr_mean"] else np.nan
        try:
            g = np.arange(edr_t[me][0], edr_t[me][-1], 1.0 / CFG.DERIVED_FS)
            if len(g) > 32:
                s = pchip_to_grid(edr_t[me], seg_edr, g)
                s = s - np.mean(s)
                f, pxx = welch(s, fs=CFG.DERIVED_FS, nperseg=min(128, len(s)))
                band = (f >= 0.1) & (f <= 0.5)        # normal breathing band
                if band.any() and pxx[band].sum() > 0:
                    out["edr_peak_freq"] = float(f[band][np.argmax(pxx[band])])
                    out["edr_power"] = band_power(f, pxx, 0.1, 0.5)
        except Exception:
            pass

    return out


print("Feature extraction defined.")

In [ ]:
# ===========================================================================
# 2.d  Process one recording end to end
# ===========================================================================

def process_record(name, base):
    """
    Full pipeline for one recording.

    Order matters:
      1. Read raw ECG and annotations.
      2. Bandpass 0.5-45 Hz for morphology.
      3. Pan-Tompkins on the filtered signal -> R peaks.
      4. Build RR and EDR series, with artefact rejection.
      5. Resample both onto a uniform 4 Hz grid spanning the whole recording.
      6. Robust per-recording normalization.
      7. Slice into one-minute segments aligned to the annotations.
      8. Extract engineered features per minute.

    Returns (X_deep, feats_df, labels, minute_index) or None.

    X_deep has shape (n_minutes, 240, 2): the RRI and EDR channels at 4 Hz.
    We feed the deep models these DERIVED series rather than raw 6000-sample
    ECG because the apnea signature lives in rhythm and respiration, not in QRS
    morphology -- and a 240-sample input trains an order of magnitude faster
    than a 6000-sample one, which matters on a single GPU.
    """
    sig, hdr = read_signal(base)
    ecg = sig[:, 0]
    fs = hdr["fs"]
    samp, sym = read_annotations(base, "apn")

    # Keep only the two symbols this dataset uses.
    valid = [i for i, s in enumerate(sym) if s in ("A", "N")]
    if len(valid) < 10:
        return None
    labels = np.array([1 if sym[i] == "A" else 0 for i in valid], dtype=np.int64)
    minute_idx = np.array([samp[i] // int(fs * 60) for i in valid], dtype=np.int64)

    # 2. Morphology bandpass.
    ecg_f = butter_bandpass(ecg, fs, CFG.ECG_BAND[0], min(CFG.ECG_BAND[1], 0.45 * fs),
                            CFG.BUTTER_ORDER)

    # 3-4. R peaks -> derived series.
    peaks = pan_tompkins(ecg_f, fs)
    rr_t, rr, edr_t, edr = rr_and_edr(ecg_f, peaks, fs)
    if len(rr) < 50:
        print(f"  {name}: only {len(rr)} usable RR intervals -- skipping")
        return None

    dur = len(ecg) / fs
    bpm = len(peaks) / (dur / 60)
    print(f"  {name}: {dur/3600:.2f} h | {len(peaks):,} R peaks ({bpm:.0f}/min) | "
          f"{len(rr):,} clean RR | {len(labels)} annotated minutes "
          f"({int(labels.sum())} apneic)")

    # 5. Uniform 4 Hz grid across the whole recording.
    grid = np.arange(0, dur, 1.0 / CFG.DERIVED_FS)
    rr_grid = pchip_to_grid(rr_t, rr, grid)
    edr_grid = pchip_to_grid(edr_t, edr, grid)

    # 6. Robust normalization, per recording.
    rr_n, edr_n = robust_z(rr_grid), robust_z(edr_grid)

    # 7-8. Slice and featurise.
    L = CFG.DERIVED_LEN
    X, rows, keep = [], [], []
    for k, mi in enumerate(minute_idx):
        s = int(mi * 60 * CFG.DERIVED_FS)
        e = s + L
        if e > len(rr_n):
            continue
        X.append(np.stack([rr_n[s:e], edr_n[s:e]], axis=1))
        f = segment_features(rr_t, rr, edr_t, edr, mi * 60.0, (mi + 1) * 60.0)
        f["minute"] = int(mi)
        # Position through the night, available to any real monitor.
        f["night_frac"] = k / max(len(minute_idx) - 1, 1)
        rows.append(f)
        keep.append(k)

    if not X:
        return None
    return (np.asarray(X, dtype=np.float16),
            pd.DataFrame(rows),
            labels[keep],
            minute_idx[keep])


print("process_record defined.")

In [ ]:
# ===========================================================================
# 2.e  ETL over all records (cached and resumable)
# ===========================================================================

def cache_path(name):
    return Path(CFG.CACHE_DIR) / f"{name}.npz"


def run_etl():
    """
    Process every record, skipping any already cached.

    Errors are caught per record and reported rather than raised, so one bad
    file cannot discard the whole run.
    """
    names = sorted(RECORDS)
    if CFG.LIMIT_RECORDS:
        # Take a sample spread ACROSS record groups (a/b/c/x), not the first N
        # alphabetically. Slicing alphabetically would give only a-records, and
        # the official test set (the x records) would come back EMPTY -- which
        # then breaks every held-out evaluation downstream.
        by_letter = {}
        for n in names:
            by_letter.setdefault(n[0], []).append(n)
        picked, i = [], 0
        while len(picked) < CFG.LIMIT_RECORDS:
            added = False
            for L in sorted(by_letter):
                if i < len(by_letter[L]) and len(picked) < CFG.LIMIT_RECORDS:
                    picked.append(by_letter[L][i])
                    added = True
            if not added:
                break
            i += 1
        names = sorted(picked)
        print(f"LIMIT_RECORDS={CFG.LIMIT_RECORDS} -> {names}\n")

    done, failed, skipped = [], [], []
    t0 = time.time()
    for i, name in enumerate(names, 1):
        cp = cache_path(name)
        if cp.exists():
            skipped.append(name)
            continue
        print(f"[{i}/{len(names)}]", end=" ", flush=True)
        try:
            res = process_record(name, RECORDS[name])
            if res is None:
                failed.append((name, "unusable"))
                continue
            X, feats, y, mins = res
            np.savez_compressed(cp, X=X, y=y, minutes=mins,
                                feat_values=feats.to_numpy(dtype=np.float32),
                                feat_names=np.array(feats.columns.tolist()),
                                record=name)
            done.append(name)
        except Exception as ex:
            print(f"  {name}: FAILED {type(ex).__name__}: {ex}")
            failed.append((name, str(ex)[:150]))
        finally:
            gc.collect()

    print(f"\nETL done in {(time.time()-t0)/60:.1f} min | new {len(done)} | "
          f"cached {len(skipped)} | failed {len(failed)}")
    for n, why in failed:
        print(f"  {n}: {why}")


run_etl()
cached = sorted(Path(CFG.CACHE_DIR).glob("*.npz"))
print(f"\nCache: {len(cached)} records, "
      f"{sum(p.stat().st_size for p in cached)/1e6:.0f} MB")

In [ ]:
# ===========================================================================
# 2.f  Assemble the cache
# ===========================================================================

def load_cache():
    """
    Concatenate all cached records.

    `groups` -- the record ID for every minute -- is the critical output. Every
    split in this notebook is made on `groups`, never on minute index.
    Consecutive minutes from one night share physiology, the spectral features
    use overlapping +/- 90 s windows, and per-recording normalization gives
    every minute from one night a shared fingerprint. Splitting on minutes would
    leak all three.
    """
    Xs, ys, gs, Fs, Ms = [], [], [], [], []
    names = None
    for p in sorted(Path(CFG.CACHE_DIR).glob("*.npz")):
        d = np.load(p, allow_pickle=True)
        rec = str(d["record"])
        Xs.append(d["X"]); ys.append(d["y"]); Ms.append(d["minutes"])
        gs.append(np.array([rec] * len(d["y"])))
        Fs.append(d["feat_values"])
        if names is None:
            names = [str(x) for x in d["feat_names"]]
    if not Xs:
        raise RuntimeError("Cache empty -- run the ETL.")
    return (np.concatenate(Xs), np.concatenate(ys), np.concatenate(gs),
            pd.DataFrame(np.concatenate(Fs), columns=names), np.concatenate(Ms))


X_deep, y, groups, FEATS, MINUTES = load_cache()

print(f"Deep tensor : {X_deep.shape}  ({X_deep.nbytes/1e6:.0f} MB float16)")
print(f"  channels  : ['RRI', 'EDR'] at {CFG.DERIVED_FS} Hz")
print(f"Features    : {FEATS.shape[1]}")
print(f"Records     : {len(np.unique(groups))}")
print(f"Minutes     : {len(y):,}  ({len(y)/60:.0f} hours)")
print(f"\nClass balance: apnea {int(y.sum()):,} ({100*y.mean():.1f}%)  |  "
      f"normal {int((1-y).sum()):,} ({100*(1-y).mean():.1f}%)")

print("\nFeature missingness (worst 8):")
print((FEATS.isna().mean() * 100).sort_values(ascending=False).head(8).round(2).to_string())

In [ ]:
# ===========================================================================
# 2.g  Sanity check -- look at the signal before trusting any model
# ===========================================================================
# Automated pipelines fail silently: a wrong corner frequency or an off-by-one
# in segmentation produces arrays of the right SHAPE full of garbage. Plotting
# is the only reliable defence.
#
# What to look for in panel (a): during apneic minutes the RRI trace should show
# a slow oscillation -- the bradycardia-tachycardia cycle -- that is absent or
# much weaker in normal minutes. That oscillation is the signal the entire study
# depends on. If you cannot see it, something upstream is broken.

fig, axes = plt.subplots(2, 2, figsize=(15, 7))
t = np.arange(CFG.DERIVED_LEN) / CFG.DERIVED_FS

for col, (lab, title) in enumerate([(0, "Normal minute"), (1, "Apneic minute")]):
    idx = np.where(y == lab)[0]
    pick = idx[len(idx) // 2]
    axes[0, col].plot(t, X_deep[pick, :, 0].astype(np.float32), lw=1.2, color="#C44E52")
    axes[0, col].set_title(f"(a) RR intervals — {title}  [{groups[pick]}]")
    axes[0, col].set_ylabel("RRI (normalised)")
    axes[1, col].plot(t, X_deep[pick, :, 1].astype(np.float32), lw=1.2, color="#4C72B0")
    axes[1, col].set_title(f"(b) ECG-derived respiration — {title}")
    axes[1, col].set_ylabel("EDR (normalised)")
    axes[1, col].set_xlabel("seconds within the minute")

plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig2_signal_check.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Feature separation: do any features differ between classes? -----------
# If no single feature separates the classes, a model that scores well is
# probably exploiting something other than physiology.
sep = []
for c in FEATS.columns:
    if c in ("minute",):
        continue
    a, b = FEATS[c][y == 1].dropna(), FEATS[c][y == 0].dropna()
    if len(a) > 30 and len(b) > 30:
        pooled = np.sqrt((a.var() + b.var()) / 2)
        if pooled > 0:
            sep.append((c, (a.mean() - b.mean()) / pooled))   # Cohen's d
sep = pd.Series(dict(sep)).sort_values(key=np.abs, ascending=False)

plt.figure(figsize=(8, 7))
top = sep.head(18)[::-1]
plt.barh(top.index, top.values,
         color=["#C44E52" if v > 0 else "#4C72B0" for v in top.values])
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Cohen's d  (apnea vs normal minutes)")
plt.title("Which features separate apneic from normal minutes?")
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig3_feature_separation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Strongest separators:")
print(sep.head(8).round(3).to_string())

---
# 3. Stage 1 — Per-minute apnea detection

Three models on identical splits:

| Model | Input | Sees neighbouring minutes? | Review source |
|---|---|---|---|
| **Random Forest** | ~28 engineered features | No | Table 7 |
| **1D-CNN** | RRI + EDR, one minute (240×2) | No | Table 8 |
| **CNN-BiLSTM** | RRI + EDR, 11-minute window | **Yes** | Table 8 |

The 1D-CNN and CNN-BiLSTM share a **byte-identical convolutional encoder**, so their performance
gap measures exactly one thing: the value of temporal context.

### Two evaluations, deliberately

**The official split.** Apnea-ECG ships with a 35-record learning set (`a`, `b`, `c`) and a
35-record test set (`x`), designed for the 2000 Computers in Cardiology Challenge and disjoint by
patient. Training on the learning set and testing on `x` is a genuine held-out evaluation on data
the model has never touched — stronger than cross-validation alone, and directly comparable to
published results on this benchmark.

**Grouped cross-validation** within the learning set, for variance estimates. Folds are split by
*record*, never by minute. Splitting on minutes would put minutes from the same night on both sides,
and since consecutive minutes share physiology and our spectral features use overlapping ±90 s
windows, that leaks badly. Much of the very high accuracy in the published literature comes from
exactly this mistake.

In [ ]:
# ===========================================================================
# 3.a  Splits
# ===========================================================================

is_test = np.array([g[0] == "x" for g in groups])
learn_idx = np.where(~is_test)[0]
test_idx = np.where(is_test)[0]

# If LIMIT_RECORDS was set and happened to exclude all x records, there is no
# official test set. Everything below degrades to cross-validation only rather
# than crashing, so a smoke test still exercises the whole pipeline.
HAVE_TEST = len(test_idx) > 0

print("Official challenge split")
print(f"  learning set : {len(learn_idx):,} minutes / "
      f"{len(np.unique(groups[learn_idx]))} records  "
      f"(apnea {100*y[learn_idx].mean():.1f}%)")
if HAVE_TEST:
    print(f"  test set     : {len(test_idx):,} minutes / "
          f"{len(np.unique(groups[test_idx]))} records  "
          f"(apnea {100*y[test_idx].mean():.1f}%)")
    assert not (set(groups[learn_idx]) & set(groups[test_idx])), "official split overlaps"
    print("  disjoint by record: OK")
else:
    print("  test set     : EMPTY")
    print("\n  No x-records were processed, so the held-out test evaluation is")
    print("  skipped and only cross-validation runs. This is expected during a")
    print("  smoke test. For the real run set LIMIT_RECORDS = None.")


def record_bounds(g):
    """Map each record to its [start, end) slice, so context windows never
    straddle two patients."""
    bounds, order, s = {}, [], 0
    for i in range(1, len(g) + 1):
        if i == len(g) or g[i] != g[s]:
            bounds[g[s]] = (s, i); order.append(g[s]); s = i
    return bounds, order


BOUNDS, REC_ORDER = record_bounds(groups)
EP_START = np.zeros(len(y), dtype=np.int64)
EP_END = np.zeros(len(y), dtype=np.int64)
for r, (a, b) in BOUNDS.items():
    EP_START[a:b] = a; EP_END[a:b] = b

# Grouped CV within the learning set only.
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(learn_idx[tr], learn_idx[te]) for tr, te in
         sgkf.split(np.zeros(len(learn_idx)), y[learn_idx], groups[learn_idx])]

print(f"\n{CFG.N_FOLDS}-fold record-grouped CV inside the learning set")
for k, (tr, te) in enumerate(FOLDS):
    assert not (set(groups[tr]) & set(groups[te])), f"LEAK in fold {k}"
    print(f"  fold {k}: train {len(tr):5,} min / {len(np.unique(groups[tr])):2d} rec | "
          f"test {len(te):5,} min / {len(np.unique(groups[te])):2d} rec | "
          f"apnea {100*y[te].mean():.1f}%")
print("Leakage check passed.")

In [ ]:
# ===========================================================================
# 3.b  Metrics
# ===========================================================================

def binary_metrics(y_true, y_pred, proba=None):
    """
    Metric set chosen to match the review's Tables 7 and 8.

    SENSITIVITY and SPECIFICITY are reported separately, not merged into
    accuracy, because they carry different clinical costs. In screening, a false
    negative (missing an apneic patient) is far worse than a false positive
    (an unnecessary sleep study). A model with 90% accuracy and 40% sensitivity
    is useless for this purpose, and only the split metrics reveal that.

    COHEN'S KAPPA corrects for the agreement expected by chance given the class
    distribution.
    """
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": recall_score(1 - y_true, 1 - y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }
    if proba is not None and len(np.unique(y_true)) > 1:
        m["auc"] = roc_auc_score(y_true, proba)
    return m


RESULTS_CV = {}      # model -> per-fold metrics (learning set)
RESULTS_TEST = {}    # model -> metrics on the official test set
OOF = {}             # model -> out-of-fold predictions (learning set)
TESTPRED = {}        # model -> (pred, proba) on the official test set
print("Metrics defined.")

In [ ]:
# ===========================================================================
# 3.c  Random Forest baseline (review Table 7)
# ===========================================================================
# A strong baseline, not a straw man: tree ensembles handle mixed-scale
# features, need no normalization, and are the review's most-used classical
# model. class_weight="balanced" prevents collapse onto the majority class.

FEATURE_COLS = [c for c in FEATS.columns if c != "minute"]
Xf = FEATS[FEATURE_COLS].to_numpy(dtype=np.float32)
print(f"Feature matrix: {Xf.shape}")


def rf_factory():
    return Pipeline([
        # Median imputation: spectral features are NaN in minutes where too few
        # clean beats survived artefact rejection. Median keeps the minute (and
        # therefore the night's continuity) rather than dropping the row.
        ("impute", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(
            n_estimators=500, max_depth=None, min_samples_leaf=4,
            class_weight="balanced", n_jobs=-1, random_state=SEED)),
    ])


# --- cross-validation on the learning set ---
oof = np.full(len(y), -1, dtype=np.int64)
oof_p = np.full(len(y), np.nan, dtype=np.float32)
fold_m = []
for k, (tr, te) in enumerate(FOLDS):
    t0 = time.time()
    mdl = rf_factory().fit(Xf[tr], y[tr])
    p = mdl.predict_proba(Xf[te])[:, 1]
    pred = (p >= 0.5).astype(int)
    oof[te], oof_p[te] = pred, p
    m = binary_metrics(y[te], pred, p)
    fold_m.append(m)
    print(f"  fold {k}: acc={m['accuracy']:.3f} sens={m['sensitivity']:.3f} "
          f"spec={m['specificity']:.3f} auc={m.get('auc', np.nan):.3f} "
          f"({time.time()-t0:.0f}s)")

RESULTS_CV["RandomForest"] = fold_m
OOF["RandomForest"] = (oof, oof_p)

# --- fit on the whole learning set, evaluate on the official test set ---
if HAVE_TEST:
    mdl = rf_factory().fit(Xf[learn_idx], y[learn_idx])
    p_test = mdl.predict_proba(Xf[test_idx])[:, 1]
    pred_test = (p_test >= 0.5).astype(int)
    RESULTS_TEST["RandomForest"] = binary_metrics(y[test_idx], pred_test, p_test)
    TESTPRED["RandomForest"] = (pred_test, p_test)
    print(f"\nRandomForest on official test set: "
          f"acc={RESULTS_TEST['RandomForest']['accuracy']:.3f}  "
          f"sens={RESULTS_TEST['RandomForest']['sensitivity']:.3f}  "
          f"spec={RESULTS_TEST['RandomForest']['specificity']:.3f}  "
          f"auc={RESULTS_TEST['RandomForest']['auc']:.3f}")
else:
    mdl = rf_factory().fit(Xf[learn_idx], y[learn_idx])
    print("\nNo official test set in this run -- skipping held-out evaluation.")

# Feature importance, diagnostic only (fitted on the learning set, not held out).
imp = pd.Series(mdl.named_steps["rf"].feature_importances_,
                index=FEATURE_COLS).sort_values(ascending=False)
plt.figure(figsize=(8, 7))
imp.head(18)[::-1].plot(kind="barh", color="#4C72B0")
plt.xlabel("Gini importance")
plt.title("Random Forest feature importance (diagnostic, not a held-out result)")
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig4_rf_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ===========================================================================
# 3.d  Datasets
# ===========================================================================

class MinuteDataset(Dataset):
    """
    Single-minute dataset for the 1D-CNN.

    Returns (channels, time) because Conv1d expects (batch, channels, length).
    The cache stores (time, channels), so we transpose per item rather than
    keeping a second copy of the array.
    """
    def __init__(self, X, y, idx, channels=None):
        self.X, self.y, self.idx = X, y, idx
        self.ch = channels if channels is not None else list(range(X.shape[2]))

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = self.X[j][:, self.ch].astype(np.float32).T
        return torch.from_numpy(np.ascontiguousarray(x)), int(self.y[j])


class WindowDataset(Dataset):
    """
    Context-window dataset for the CNN-BiLSTM.

    Returns the 11 minutes centred on minute j, with indices CLAMPED to that
    record's own range. Without the clamp, a window near the start of one
    recording would reach back into the previous patient's night -- a different
    person, different electrodes, different disease.

    The label is that of the CENTRE minute only: we classify one minute using
    its neighbours as context, not sequence-to-sequence.
    """
    def __init__(self, X, y, idx, s, e, ctx, channels=None):
        self.X, self.y, self.idx = X, y, idx
        self.s, self.e, self.ctx = s, e, ctx
        self.ch = channels if channels is not None else list(range(X.shape[2]))

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        w = np.clip(np.arange(j - self.ctx, j + self.ctx + 1),
                    self.s[j], self.e[j] - 1)
        x = self.X[w][:, :, self.ch].astype(np.float32)     # (W, T, C)
        x = np.transpose(x, (0, 2, 1))                      # (W, C, T)
        return torch.from_numpy(np.ascontiguousarray(x)), int(self.y[j])


print("Datasets defined.")

In [ ]:
# ===========================================================================
# 3.e  Architectures
# ===========================================================================

class ConvEncoder(nn.Module):
    """
    The shared encoder. THIS IS THE CONTROLLED VARIABLE -- both models use
    exactly this module, so any difference between them comes from the
    sequence layer alone.

    Design notes:
      * Kernels shrink with depth (7 -> 5 -> 3). At 4 Hz a width-7 kernel spans
        1.75 s, enough to see part of one respiratory cycle; deeper narrow
        kernels combine those primitives into longer patterns.
      * BatchNorm after every convolution keeps the RRI and EDR channels on
        comparable scales, which matters because they have quite different
        statistics even after robust normalization.
      * MaxPool rather than strided convolution: an apnea-related dip a second
        earlier or later within the minute means the same thing, so we want
        translation tolerance.
      * AdaptiveAvgPool1d(1) collapses time to a fixed embedding, making the
        encoder agnostic to input length.
    """
    def __init__(self, in_ch, emb=128, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),                                   # 240 -> 120
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),                                   # 120 -> 60
            nn.Dropout(p),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.MaxPool1d(2),                                   # 60 -> 30
            nn.Conv1d(128, emb, 3, padding=1), nn.BatchNorm1d(emb), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.emb = emb

    def forward(self, x):
        return self.net(x).squeeze(-1)


class CNN1D(nn.Module):
    """One minute in isolation. The no-context control condition."""
    def __init__(self, in_ch, emb=128, p=0.3):
        super().__init__()
        self.encoder = ConvEncoder(in_ch, emb, p)
        self.head = nn.Sequential(nn.Linear(emb, 64), nn.ReLU(),
                                  nn.Dropout(p), nn.Linear(64, 2))

    def forward(self, x):
        return self.head(self.encoder(x))


class CNNBiLSTM(nn.Module):
    """
    Same encoder applied to each of 11 minutes, then a bidirectional LSTM
    across those embeddings, classifying the centre minute.

    Why BIDIRECTIONAL: apnea context is not causal. Knowing the NEXT few
    minutes are apneic is as informative about the current minute as knowing the
    previous ones were. We are analysing a recorded night offline, not
    predicting in real time, so discarding the future would waste half the
    available context for nothing.

    Why this should help: apneas come in runs. A single ambiguous minute -- mild
    rate variation that could be apnea or could be a position change -- is
    resolved by whether it sits inside a run of clearly apneic minutes. Only the
    sequence model can use that.
    """
    def __init__(self, in_ch, emb=128, hidden=64, layers=2, p=0.3):
        super().__init__()
        self.encoder = ConvEncoder(in_ch, emb, p)
        self.lstm = nn.LSTM(emb, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=p if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(),
                                  nn.Dropout(p), nn.Linear(64, 2))

    def forward(self, x):                      # (B, W, C, T)
        B, W, C, T = x.shape
        # Fold the window into the batch so all B*W minutes encode in one pass.
        z = self.encoder(x.reshape(B * W, C, T)).reshape(B, W, -1)
        out, _ = self.lstm(z)
        return self.head(out[:, W // 2, :])    # classify the centre minute


for nm, m in [("1D-CNN", CNN1D(2)), ("CNN-BiLSTM", CNNBiLSTM(2))]:
    print(f"{nm:12s}: {sum(p.numel() for p in m.parameters()):,} parameters")
print(f"{'(encoder)':12s}: {sum(p.numel() for p in ConvEncoder(2).parameters()):,} "
      f"parameters — shared identically by both")

In [ ]:
# ===========================================================================
# 3.f  Training
# ===========================================================================

def class_weights(y_tr):
    """Inverse-frequency weights, normalised to mean 1 so the loss scale is stable."""
    c = np.bincount(y_tr, minlength=2).astype(np.float64)
    w = c.sum() / (2.0 * np.maximum(c, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def split_train_val(train_idx, g, frac=0.2, seed=SEED):
    """Carve a RECORD-disjoint validation set out of the training fold."""
    recs = np.unique(g[train_idx])
    rng = np.random.RandomState(seed); rng.shuffle(recs)
    val = set(recs[:max(2, int(len(recs) * frac))])
    m = np.array([r in val for r in g[train_idx]])
    return train_idx[~m], train_idx[m]


def train_fold(model, tr_ds, va_ds, te_ds, w, tag=""):
    """
    Train with early stopping on validation F1.

    * WEIGHTED CROSS-ENTROPY, because apnea is the minority class and unweighted
      training drifts toward predicting "normal" everywhere.
    * EARLY STOPPING ON F1, not loss. Loss is dominated by the majority class,
      so a model can reduce loss while getting worse at detecting apnea.
    * THE VALIDATION SET IS RECORD-DISJOINT and carved from the TRAINING fold,
      never the test fold. Choosing the stopping epoch with test data is a real
      if subtle leak.
    * AdamW for decoupled weight decay, which regularises correctly alongside
      adaptive learning rates.
    """
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR,
                            weight_decay=CFG.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.EPOCHS)
    crit = nn.CrossEntropyLoss(weight=w.to(DEVICE))

    tl = DataLoader(tr_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(va_ds, batch_size=CFG.BATCH_SIZE * 2, num_workers=2)
    sl = DataLoader(te_ds, batch_size=CFG.BATCH_SIZE * 2, num_workers=2)

    best, best_state, bad = -1.0, None, 0
    for ep in range(CFG.EPOCHS):
        model.train(); tot = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            # Clip gradients: an LSTM over 11 steps can occasionally produce a
            # large gradient that would undo a converged model in one batch.
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); tot += loss.item() * len(yb)
        sched.step()

        model.eval(); vp, vt = [], []
        with torch.no_grad():
            for xb, yb in vl:
                vp.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
                vt.append(yb.numpy())
        vf1 = f1_score(np.concatenate(vt), np.concatenate(vp), zero_division=0)

        flag = ""
        if vf1 > best:
            best, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            flag = " *"
        else:
            bad += 1
        if (ep + 1) % 5 == 0 or flag:
            print(f"    {tag} ep{ep+1:02d} loss={tot/max(len(tl.dataset),1):.4f} "
                  f"val_f1={vf1:.4f}{flag}")
        if bad >= CFG.PATIENCE:
            print(f"    early stop @ {ep+1} (best val F1 {best:.4f})")
            break

    if best_state:
        model.load_state_dict(best_state)

    model.eval(); pr, pb, tt = [], [], []
    with torch.no_grad():
        for xb, yb in sl:
            lg = model(xb.to(DEVICE))
            pb.append(F.softmax(lg, 1)[:, 1].cpu().numpy())
            pr.append(lg.argmax(1).cpu().numpy())
            tt.append(yb.numpy())
    return np.concatenate(pr), np.concatenate(pb), np.concatenate(tt), model


def run_deep(name, factory, sequence=False, channels=None):
    """Cross-validate, then refit on the full learning set and score the
    official test set."""
    n_ch = len(channels) if channels else X_deep.shape[2]
    mk = (lambda idx: WindowDataset(X_deep, y, idx, EP_START, EP_END,
                                    CFG.CONTEXT, channels)) if sequence else \
         (lambda idx: MinuteDataset(X_deep, y, idx, channels))

    oof = np.full(len(y), -1, dtype=np.int64)
    oof_p = np.full(len(y), np.nan, dtype=np.float32)
    fold_m = []
    for k, (tr_all, te) in enumerate(FOLDS):
        print(f"  --- fold {k} ---")
        t0 = time.time()
        tr, va = split_train_val(tr_all, groups)
        pred, prob, true, _ = train_fold(factory(n_ch), mk(tr), mk(va), mk(te),
                                         class_weights(y[tr]), tag=f"f{k}")
        oof[te], oof_p[te] = pred, prob
        m = binary_metrics(true, pred, prob)
        fold_m.append(m)
        print(f"    fold {k}: acc={m['accuracy']:.3f} sens={m['sensitivity']:.3f} "
              f"spec={m['specificity']:.3f} auc={m.get('auc', np.nan):.3f} "
              f"({(time.time()-t0)/60:.1f} min)")
        torch.cuda.empty_cache(); gc.collect()

    RESULTS_CV[name] = fold_m
    OOF[name] = (oof, oof_p)

    if not HAVE_TEST:
        print("  (no official test set in this run -- CV only)")
        return

    print(f"  --- refit on full learning set -> official test set ---")
    tr, va = split_train_val(learn_idx, groups)
    pred, prob, true, _ = train_fold(factory(n_ch), mk(tr), mk(va), mk(test_idx),
                                     class_weights(y[tr]), tag="final")
    RESULTS_TEST[name] = binary_metrics(true, pred, prob)
    TESTPRED[name] = (pred, prob)
    t = RESULTS_TEST[name]
    print(f"\n{name} TEST SET: acc={t['accuracy']:.3f} sens={t['sensitivity']:.3f} "
          f"spec={t['specificity']:.3f} auc={t['auc']:.3f}")
    torch.cuda.empty_cache(); gc.collect()


print("Training utilities defined.")

In [ ]:
print("=" * 70); print("1D-CNN  (single minute, no context)"); print("=" * 70)
run_deep("1D-CNN", lambda c: CNN1D(c), sequence=False)

In [ ]:
print("=" * 70)
print("CNN-BiLSTM  (11-minute context, identical encoder)")
print("=" * 70)
run_deep("CNN-BiLSTM", lambda c: CNNBiLSTM(c), sequence=True)

In [ ]:
# ===========================================================================
# 3.g  Ablation: does ECG-derived respiration add anything beyond rhythm?
# ===========================================================================
# The RRI channel carries rhythm; the EDR channel carries a respiration
# surrogate. Both are derived from the same R peaks, so it is a fair question
# whether EDR contributes independent information or is redundant.
#
# Strict subtraction: same architecture, same folds, same seed, same
# hyperparameters. Only channel 1 (EDR) is removed. Any difference is
# attributable to EDR.
#
# A null result would be informative too: it would say the respiratory
# modulation is already recoverable from rhythm alone.

print("=" * 70); print("ABLATION: CNN-BiLSTM with RRI only (no EDR)"); print("=" * 70)
run_deep("CNN-BiLSTM (RRI only)", lambda c: CNNBiLSTM(c), sequence=True, channels=[0])

In [ ]:
# ===========================================================================
# 3.h  Stage 1 results
# ===========================================================================

print("=" * 78)
print("CROSS-VALIDATION (learning set, record-grouped, mean ± SD over folds)")
print("=" * 78)
cv_rows = []
for nm, folds in RESULTS_CV.items():
    d = pd.DataFrame(folds)
    cv_rows.append({"Model": nm, **{c: f"{d[c].mean():.3f} ± {d[c].std():.3f}"
                                    for c in d.columns}})
CV_TABLE = pd.DataFrame(cv_rows)
display(CV_TABLE)

print("\n" + "=" * 78)
if HAVE_TEST:
    print("OFFICIAL HELD-OUT TEST SET (35 records never seen in training)")
    print("=" * 78)
    TEST_TABLE = pd.DataFrame([{"Model": nm, **m} for nm, m in RESULTS_TEST.items()])
    display(TEST_TABLE.round(4))
    TEST_TABLE.round(4).to_csv(Path(CFG.OUT_DIR) / "table2_test.csv", index=False)
else:
    print("OFFICIAL TEST SET: not available in this run")
    print("=" * 78)
    TEST_TABLE = pd.DataFrame()

CV_TABLE.to_csv(Path(CFG.OUT_DIR) / "table1_cv.csv", index=False)

# --- The headline comparison ---
if "1D-CNN" in RESULTS_CV and "CNN-BiLSTM" in RESULTS_CV:
    a = pd.DataFrame(RESULTS_CV["1D-CNN"])["f1"]
    b = pd.DataFrame(RESULTS_CV["CNN-BiLSTM"])["f1"]
    d = b.values - a.values
    print("\n" + "=" * 66)
    print("SECONDARY QUESTION: does temporal context improve detection?")
    print("=" * 66)
    print(f"  1D-CNN      F1 (CV): {a.mean():.4f} ± {a.std():.4f}")
    print(f"  CNN-BiLSTM  F1 (CV): {b.mean():.4f} ± {b.std():.4f}")
    print(f"  paired difference  : {d.mean():+.4f} ± {d.std():.4f}  "
          f"({int((d > 0).sum())}/{len(d)} folds improved)")
    if HAVE_TEST and "1D-CNN" in RESULTS_TEST and "CNN-BiLSTM" in RESULTS_TEST:
        print(f"\n  On the held-out test set:")
        print(f"    1D-CNN     F1 = {RESULTS_TEST['1D-CNN']['f1']:.4f}, "
              f"AUC = {RESULTS_TEST['1D-CNN']['auc']:.4f}")
        print(f"    CNN-BiLSTM F1 = {RESULTS_TEST['CNN-BiLSTM']['f1']:.4f}, "
              f"AUC = {RESULTS_TEST['CNN-BiLSTM']['auc']:.4f}")

if "CNN-BiLSTM (RRI only)" in RESULTS_CV:
    print("\n" + "=" * 66)
    print("ABLATION: contribution of ECG-derived respiration")
    print("=" * 66)
    full_cv = pd.DataFrame(RESULTS_CV["CNN-BiLSTM"])["f1"]
    rri_cv = pd.DataFrame(RESULTS_CV["CNN-BiLSTM (RRI only)"])["f1"]
    print(f"  RRI + EDR (CV F1): {full_cv.mean():.4f} ± {full_cv.std():.4f}")
    print(f"  RRI only  (CV F1): {rri_cv.mean():.4f} ± {rri_cv.std():.4f}")
    print(f"  difference       : {(full_cv.values-rri_cv.values).mean():+.4f}")

In [ ]:
# ===========================================================================
# 3.i  Confusion matrices, ROC, and an example night
# ===========================================================================

# If the official test set is unavailable, fall back to out-of-fold predictions
# on the learning set so the figures still render during a smoke test.
if HAVE_TEST:
    EVAL_IDX = test_idx
    EVAL_PRED = {nm: TESTPRED[nm] for nm in TESTPRED}
    EVAL_LABEL = "official test set"
else:
    EVAL_IDX = learn_idx
    EVAL_PRED = {nm: (OOF[nm][0][learn_idx], OOF[nm][1][learn_idx]) for nm in OOF}
    EVAL_LABEL = "out-of-fold (learning set)"

names = list(EVAL_PRED.keys())
fig, axes = plt.subplots(1, len(names), figsize=(4.3 * len(names), 4))
if len(names) == 1:
    axes = [axes]
for ax, nm in zip(axes, names):
    cm = confusion_matrix(y[EVAL_IDX], EVAL_PRED[nm][0], normalize="true")
    sns.heatmap(cm, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=["Normal", "Apnea"], yticklabels=["Normal", "Apnea"],
                ax=ax, cbar=False)
    ax.set_title(nm, fontsize=10); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle(f"{EVAL_LABEL} — row-normalised (diagonal = sensitivity / specificity)")
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig5_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6.6, 6))
for nm, (pred, prob) in EVAL_PRED.items():
    ok = np.isfinite(prob)
    if ok.sum() < 10 or len(np.unique(y[EVAL_IDX][ok])) < 2:
        continue
    fpr, tpr, _ = roc_curve(y[EVAL_IDX][ok], prob[ok])
    plt.plot(fpr, tpr, lw=2,
             label=f"{nm} (AUC = {roc_auc_score(y[EVAL_IDX][ok], prob[ok]):.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
plt.xlabel("1 − Specificity"); plt.ylabel("Sensitivity")
plt.title(f"Per-minute apnea detection — {EVAL_LABEL}")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig6_roc.png", dpi=150, bbox_inches="tight")
plt.show()

# --- One night, true vs predicted ---
scoreboard = RESULTS_TEST if HAVE_TEST else {
    nm: pd.DataFrame(f).mean().to_dict() for nm, f in RESULTS_CV.items()}
best = max(scoreboard, key=lambda k: scoreboard[k]["f1"])
eval_recs = sorted(set(groups[EVAL_IDX]))
rec = eval_recs[len(eval_recs) // 2]
s, e = BOUNDS[rec]
pos = np.searchsorted(EVAL_IDX, np.arange(s, e))
pos = pos[pos < len(EVAL_IDX)]
hrs = np.arange(e - s) / 60.0

fig, axes = plt.subplots(2, 1, figsize=(14, 4.2), sharex=True, sharey=True)
axes[0].fill_between(hrs, 0, y[s:e], step="post", color="#2c3e50", alpha=0.85)
axes[0].set_title(f"Record {rec} — expert annotation")
axes[1].fill_between(hrs, 0, EVAL_PRED[best][0][pos][:e - s], step="post",
                     color="#C44E52", alpha=0.85)
axes[1].set_title(f"Predicted from ECG alone — {best}")
for ax in axes:
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "Apnea"])
axes[1].set_xlabel("Hours from recording start")
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig7_night.png", dpi=150, bbox_inches="tight")
plt.show()

---
# 4. Stage 2 — Who needs a sleep study?

Stage 1 answers "is this minute apneic". The clinical question is "does this **patient** need
referral", so we aggregate.

For each recording we count the predicted apneic minutes and form an **estimated apnea index**
(apneic minutes per hour). Two things follow:

1. **Classification.** Flag recordings with at least 100 predicted apneic minutes — the Apnea-ECG
   challenge's own Class A criterion. We use the dataset's definition rather than inventing one.
2. **Agreement.** Compare the estimated apnea index against the reference index derived from expert
   annotations, by correlation and Bland–Altman.

This stage needs no separate training. It is a deterministic aggregation of Stage 1's output, which
means it introduces no additional opportunity to overfit — worth stating, because n = 70 recordings
is small and any fitted per-recording model would carry wide uncertainty.

In [ ]:
# ===========================================================================
# 4.a  Aggregate to the recording level
# ===========================================================================

def recording_table(model_name):
    """
    Aggregate per-minute predictions into per-recording severity.

    Uses out-of-fold predictions for learning-set records and held-out test
    predictions for test records, so every value is from a model that never saw
    that recording in training.
    """
    oof_pred, _ = OOF[model_name]

    full = np.full(len(y), -1, dtype=np.int64)
    full[learn_idx] = oof_pred[learn_idx]
    if HAVE_TEST and model_name in TESTPRED:
        full[test_idx] = TESTPRED[model_name][0]

    rows = []
    for rec, (s, e) in BOUNDS.items():
        pr, tr = full[s:e], y[s:e]
        if (pr < 0).all():
            continue
        n_min = len(tr)
        hours = n_min / 60.0
        rows.append(dict(
            record=rec,
            group="test" if rec[0] == "x" else "learning",
            n_minutes=n_min,
            true_apnea_min=int(tr.sum()),
            pred_apnea_min=int(pr.sum()),
            true_index=60.0 * tr.sum() / n_min,
            pred_index=60.0 * pr.sum() / n_min,
            true_apneic=int(tr.sum() >= CFG.APNEA_MINUTES_THRESHOLD),
            pred_apneic=int(pr.sum() >= CFG.APNEA_MINUTES_THRESHOLD),
        ))
    return pd.DataFrame(rows).set_index("record")


_board = RESULTS_TEST if HAVE_TEST else {
    nm: pd.DataFrame(f).mean().to_dict() for nm, f in RESULTS_CV.items()}
best_model = max(_board, key=lambda k: _board[k]["f1"])
print(f"Aggregating with the best Stage 1 model: {best_model}\n")
REC = recording_table(best_model)

acc = (REC["true_apneic"] == REC["pred_apneic"]).mean()
sens = recall_score(REC["true_apneic"], REC["pred_apneic"], zero_division=0)
spec = recall_score(1 - REC["true_apneic"], 1 - REC["pred_apneic"], zero_division=0)
r = np.corrcoef(REC["true_index"], REC["pred_index"])[0, 1]
mae = mean_absolute_error(REC["true_index"], REC["pred_index"])

print("=" * 66)
print(f"PER-RECORDING SCREENING  (n = {len(REC)})")
print("=" * 66)
print(f"  accuracy    : {acc:.3f}  ({int((REC['true_apneic']==REC['pred_apneic']).sum())}/{len(REC)})")
print(f"  sensitivity : {sens:.3f}   (apneic patients correctly flagged)")
print(f"  specificity : {spec:.3f}   (normal patients correctly cleared)")
print(f"\n  apnea index: r = {r:.3f}, MAE = {mae:.2f} apneic minutes/hour")

for grp in ["learning", "test"]:
    sub = REC[REC["group"] == grp]
    if len(sub):
        a = (sub["true_apneic"] == sub["pred_apneic"]).mean()
        rr = np.corrcoef(sub["true_index"], sub["pred_index"])[0, 1]
        print(f"  {grp:9s}: n={len(sub):2d}  accuracy={a:.3f}  r={rr:.3f}")

display(REC.head(12).round(2))
REC.round(4).to_csv(Path(CFG.OUT_DIR) / "table3_recording_level.csv")

In [ ]:
# ===========================================================================
# 4.b  Agreement figures
# ===========================================================================
# Panel (a) is the headline clinical figure: predicted vs true apnea burden,
# one point per patient. Points near the diagonal mean the estimate is right.
#
# Panel (b) is a Bland-Altman plot, which is the standard way to show agreement
# between two measurement methods in clinical research. A correlation can be
# high while the method is systematically biased; Bland-Altman exposes that,
# because a non-zero mean difference is visible immediately as an offset from
# the centre line.

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6))

colors = {"learning": "#4C72B0", "test": "#C44E52"}
for grp, sub in REC.groupby("group"):
    axes[0].scatter(sub["true_index"], sub["pred_index"], s=55, alpha=0.8,
                    color=colors.get(grp, "grey"), label=f"{grp} (n={len(sub)})",
                    edgecolor="white")
lim = max(REC["true_index"].max(), REC["pred_index"].max()) * 1.08
axes[0].plot([0, lim], [0, lim], "k--", lw=1.2, label="perfect agreement")
thr_idx = 60.0 * CFG.APNEA_MINUTES_THRESHOLD / REC["n_minutes"].median()
axes[0].axhline(thr_idx, color="grey", ls=":", lw=1)
axes[0].axvline(thr_idx, color="grey", ls=":", lw=1)
axes[0].set_xlabel("True apnea index (apneic min/hour, expert)")
axes[0].set_ylabel("Predicted apnea index (from ECG alone)")
axes[0].set_title(f"(a) Predicted vs true apnea burden  (r = {r:.3f})")
axes[0].legend(fontsize=9)

mean_v = (REC["true_index"] + REC["pred_index"]) / 2
diff_v = REC["pred_index"] - REC["true_index"]
bias, sd = diff_v.mean(), diff_v.std()
axes[1].scatter(mean_v, diff_v, s=55, alpha=0.8, color="#55A868", edgecolor="white")
axes[1].axhline(bias, color="crimson", lw=1.6, label=f"bias = {bias:+.2f}")
axes[1].axhline(bias + 1.96 * sd, color="crimson", ls="--", lw=1,
                label=f"95% limits ±{1.96*sd:.2f}")
axes[1].axhline(bias - 1.96 * sd, color="crimson", ls="--", lw=1)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Mean of true and predicted index")
axes[1].set_ylabel("Predicted − True")
axes[1].set_title("(b) Bland–Altman agreement")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig8_agreement.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Bias {bias:+.2f} apneic min/hour, 95% limits of agreement "
      f"[{bias-1.96*sd:+.2f}, {bias+1.96*sd:+.2f}]")
if abs(bias) > 2:
    direction = "over" if bias > 0 else "under"
    print(f"NOTE: the model systematically {direction}estimates apnea burden. "
          f"Worth reporting explicitly rather than only quoting the correlation.")

In [ ]:
# ===========================================================================
# 4.c  Screening operating point
# ===========================================================================
# A screening test should favour sensitivity: missing an apneic patient costs
# far more than an unnecessary sleep study. So instead of reporting only the
# default 0.5 cut-off, we find the per-minute threshold whose aggregated
# per-recording decision reaches >= 90% sensitivity, and report the specificity
# it buys -- i.e. what fraction of healthy people are spared a sleep study.

_, oof_prob = OOF[best_model]
full_prob = np.full(len(y), np.nan, dtype=np.float32)
full_prob[learn_idx] = oof_prob[learn_idx]
if HAVE_TEST and best_model in TESTPRED:
    full_prob[test_idx] = TESTPRED[best_model][1]

rows = []
for thr in np.arange(0.10, 0.91, 0.02):
    tp = fp = fn = tn = 0
    for rec, (s, e) in BOUNDS.items():
        p = full_prob[s:e]
        if np.isnan(p).all():
            continue
        pred_min = int(np.nansum(p >= thr))
        pred_pos = pred_min >= CFG.APNEA_MINUTES_THRESHOLD
        true_pos = int(y[s:e].sum()) >= CFG.APNEA_MINUTES_THRESHOLD
        if pred_pos and true_pos:   tp += 1
        elif pred_pos and not true_pos: fp += 1
        elif not pred_pos and true_pos: fn += 1
        else: tn += 1
    rows.append(dict(threshold=thr,
                     sensitivity=tp / max(tp + fn, 1),
                     specificity=tn / max(tn + fp, 1),
                     accuracy=(tp + tn) / max(tp + tn + fp + fn, 1)))

SWEEP = pd.DataFrame(rows)
ok = SWEEP[SWEEP["sensitivity"] >= 0.90]
print("=" * 66)
print("SCREENING OPERATING POINT")
print("=" * 66)
if len(ok):
    best_op = ok.loc[ok["specificity"].idxmax()]
    print(f"  per-minute threshold : {best_op['threshold']:.2f}")
    print(f"  sensitivity          : {best_op['sensitivity']:.3f}")
    print(f"  specificity          : {best_op['specificity']:.3f}")
    print(f"\n  At ~90% sensitivity, this model would spare "
          f"{100*best_op['specificity']:.0f}% of non-apneic patients an "
          f"unnecessary sleep study.")
else:
    print("  No threshold reached 90% sensitivity. Best available:")
    print(SWEEP.loc[SWEEP['sensitivity'].idxmax()].round(3).to_string())

plt.figure(figsize=(7.5, 5))
plt.plot(SWEEP["threshold"], SWEEP["sensitivity"], lw=2, label="Sensitivity")
plt.plot(SWEEP["threshold"], SWEEP["specificity"], lw=2, label="Specificity")
plt.plot(SWEEP["threshold"], SWEEP["accuracy"], lw=1.4, ls="--", label="Accuracy")
plt.axhline(0.90, color="grey", ls=":", lw=1)
plt.xlabel("Per-minute decision threshold")
plt.ylabel("Per-recording performance")
plt.title("Choosing an operating point for screening")
plt.legend()
plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig9_operating_point.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ===========================================================================
# 5.  Export everything for the poster
# ===========================================================================

with pd.ExcelWriter(Path(CFG.OUT_DIR) / "results.xlsx") as xl:
    CV_TABLE.to_excel(xl, sheet_name="stage1_cv", index=False)
    if not TEST_TABLE.empty:
        TEST_TABLE.round(4).to_excel(xl, sheet_name="stage1_test", index=False)
    REC.round(4).to_excel(xl, sheet_name="recording_level")
    SWEEP.round(4).to_excel(xl, sheet_name="threshold_sweep", index=False)
    COHORT.round(3).to_excel(xl, sheet_name="cohort")

print("Saved:")
for p in sorted(Path(CFG.OUT_DIR).glob("*")):
    print(f"  {p.name:38s} {p.stat().st_size/1024:8.1f} KB")

print("\n" + "=" * 70)
print("HEADLINE NUMBERS FOR THE POSTER")
print("=" * 70)
t = _board[best_model]
print(f"  Cohort            : {len(COHORT)} recordings, "
      f"{COHORT['n_minutes'].sum():,} expert-annotated minutes")
print(f"  Best model        : {best_model}")
print(f"  Per-minute ({'held-out test set' if HAVE_TEST else 'cross-validated'}):")
print(f"    accuracy        : {t['accuracy']:.1%}")
print(f"    sensitivity     : {t['sensitivity']:.1%}")
print(f"    specificity     : {t['specificity']:.1%}")
print(f"    AUC             : {t['auc']:.3f}")
print(f"  Per-recording screening:")
print(f"    accuracy        : {acc:.1%}")
print(f"    apnea index r   : {r:.3f}")

---
# 5. RQ1 — Does the ECG beat the patient's chart?

Everything so far used the 2000 challenge's own criterion, "at least 100 apneic
minutes". We now switch to the **clinical** endpoint, AHI ≥ 15 (moderate-to-severe
obstructive sleep apnea, the point where treatment is indicated), and bring in the
demographics the dataset ships but we have not yet used.

`additional-information.txt` carries age, sex, height and weight for every record,
plus the clinician-measured AHI. That lets us ask the question that actually matters:
**does one ECG lead tell us anything a patient's chart does not?**

Age, sex and BMI are free and already in every chart — they are what questionnaire
tools like STOP-BANG run on. Any new test has to beat that baseline to be worth
performing.

In [ ]:
import os, re, glob, json, pickle, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score,
                             recall_score, f1_score)

# ---------------------------------------------------------------------------
# 0. Preflight — fail here with a clear message rather than deep in the code
# ---------------------------------------------------------------------------
_need = {
    "REC": "section 4 (recording-level table)",
    "RESULTS_CV": "section 3 (model cells)",
    "X_deep": "section 2.f (assemble cache)",
    "y": "section 2.f", "groups": "section 2.f",
    "EP_START": "section 3.a (splits)", "EP_END": "section 3.a",
    "RECORDS": "section 1.b (locate records)",
    "CNNBiLSTM": "section 3.e", "WindowDataset": "section 3.d",
    "train_fold": "section 3.f", "split_train_val": "section 3.f",
    "class_weights": "section 3.f", "CFG": "section 0",
}
_missing = [k for k in _need if k not in globals()]
if _missing:
    raise NameError(
        "Missing from memory:\n" +
        "\n".join(f"  {k:16s} <- run {_need[k]}" for k in _missing) +
        "\n\nThese live in the kernel, not on disk. If the session restarted,\n"
        "Run All from the top — the ETL cache makes that fast."
    )
print("Preflight OK.")
print(f"  models trained : {list(RESULTS_CV.keys())}")
print(f"  minutes        : {len(y):,}   records: {len(np.unique(groups))}")

ARTIFACTS = Path("/kaggle/working/model"); ARTIFACTS.mkdir(parents=True, exist_ok=True)
SAMPLES = Path("/kaggle/working/samples"); SAMPLES.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# 1. Demographics
# ---------------------------------------------------------------------------
# additional-information.txt ships with Apnea-ECG and carries age, sex, height,
# weight and the clinician-measured AHI for every record. Its header spans two
# lines, so we keep only rows whose first token is a valid record name.
print("\n" + "=" * 74)
print("1. DEMOGRAPHICS")
print("=" * 74)

_hits = glob.glob(os.path.join(CFG.INPUT_ROOT, "**", "additional-information.txt"),
                  recursive=True)
if not _hits:
    raise FileNotFoundError(
        "additional-information.txt not found under /kaggle/input. It ships "
        "with the Apnea-ECG dataset — check the attached inputs.")

_rows = []
with open(_hits[0], errors="ignore") as fh:
    for line in fh:
        p = line.split()
        if len(p) < 12 or not re.fullmatch(r"[abcx]\d{2}", p[0]):
            continue
        try:
            _rows.append(dict(record=p[0], AI=float(p[5]), HI=float(p[6]),
                              AHI=float(p[7]), age=float(p[8]),
                              sex=p[9].upper()[0], height_cm=float(p[10]),
                              weight_kg=float(p[11])))
        except (ValueError, IndexError):
            continue

DEMO = pd.DataFrame(_rows).set_index("record")
DEMO["BMI"] = DEMO["weight_kg"] / (DEMO["height_cm"] / 100.0) ** 2
DEMO["is_male"] = (DEMO["sex"] == "M").astype(float)

print(f"Read {_hits[0]}")
print(f"  {len(DEMO)} records | age {DEMO.age.min():.0f}-{DEMO.age.max():.0f} "
      f"(mean {DEMO.age.mean():.1f}) | BMI mean {DEMO.BMI.mean():.1f} | "
      f"sex {DEMO.sex.value_counts().to_dict()}")
print("\nClinical severity bands:")
print(pd.cut(DEMO.AHI, [-0.01, 5, 15, 30, np.inf],
             labels=["Normal (<5)", "Mild (5-15)", "Moderate (15-30)",
                     "Severe (>=30)"]).value_counts().sort_index().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 2. Merge, drop the duplicate, set the clinical endpoint
# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("2. STAGE 2 DESIGN MATRIX")
print("=" * 74)

S2 = REC.join(DEMO[["age", "sex", "is_male", "BMI", "AI", "HI", "AHI"]], how="inner")

# c05 and c06 come from the SAME original recording (c05 begins 80 s later).
# Keeping both would place one patient on both sides of a split.
if "c05" in S2.index:
    S2 = S2.drop(index="c05")
    print("Dropped c05 (duplicate recording of c06)")

# Switch from the 2000 challenge's dataset-specific "100 apneic minutes" rule
# to the standard clinical threshold a sleep physician acts on.
AHI_THRESHOLD = 15.0
S2["y_clinical"] = (S2["AHI"] >= AHI_THRESHOLD).astype(int)
yv = S2["y_clinical"].to_numpy().astype(int)

print(f"  patients : {len(S2)}")
print(f"  positive at AHI >= {AHI_THRESHOLD:g} : {int(yv.sum())} "
      f"({100*yv.mean():.0f}%)")
print(f"  ECG-predicted apnea index vs true AHI: "
      f"r = {np.corrcoef(S2.pred_index, S2.AHI)[0,1]:.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# 3. RQ1 — does the ECG beat the patient's chart?
# ---------------------------------------------------------------------------
FEATURE_SETS = {
    "Demographics (age, sex, BMI)": ["age", "is_male", "BMI"],
    "ECG only":                     ["pred_index"],
    "ECG + demographics":           ["pred_index", "age", "is_male", "BMI"],
}
MODELS = {
    # Heavily regularised: with ~34 patients an unconstrained model memorises
    # the cohort rather than learning anything generalisable.
    "LogReg": lambda: LogisticRegression(C=0.5, class_weight="balanced",
                                         max_iter=2000, random_state=SEED),
    "RF": lambda: RandomForestClassifier(n_estimators=500, max_depth=3,
                                         min_samples_leaf=4,
                                         class_weight="balanced",
                                         random_state=SEED, n_jobs=-1),
}


def _loso(X, yv, factory):
    """
    Leave-one-subject-out CV.

    With ~34 patients, a 5-fold split leaves ~7 per test fold and an AUC on 7
    patients is far too noisy to read. LOSO uses each patient as a test case
    exactly once and pools the held-out predictions into one ROC.

    The imputer and scaler are fitted INSIDE each fold. Fitting them on the
    full cohort would leak the held-out patient into its own normalisation.
    """
    n = len(yv); proba = np.zeros(n)
    for i in range(n):
        tr = np.ones(n, bool); tr[i] = False
        pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("clf", factory())]).fit(X[tr], yv[tr])
        proba[i] = pipe.predict_proba(X[i:i + 1])[0, 1]
    pred = (proba >= 0.5).astype(int)
    res = dict(auc=roc_auc_score(yv, proba), accuracy=accuracy_score(yv, pred),
               sensitivity=recall_score(yv, pred, zero_division=0),
               specificity=recall_score(1 - yv, 1 - pred, zero_division=0),
               f1=f1_score(yv, pred, zero_division=0))
    # Bootstrap CI: resample patients with replacement. At this n it will be
    # wide, and reporting that honestly is the point.
    rng = np.random.RandomState(SEED); a = []
    for _ in range(2000):
        idx = rng.randint(0, n, n)
        if len(np.unique(yv[idx])) > 1:
            a.append(roc_auc_score(yv[idx], proba[idx]))
    res["auc_lo"], res["auc_hi"] = np.percentile(a, [2.5, 97.5])
    return res, proba


print("\n" + "=" * 74)
print(f"3. RQ1 — leave-one-subject-out, n = {len(yv)}")
print("=" * 74)
_rows, probas = [], {}
for fs, cols in FEATURE_SETS.items():
    X = S2[cols].to_numpy(dtype=np.float64)
    for mn, fac in MODELS.items():
        r, p = _loso(X, yv, fac)
        r.update(features=fs, model=mn)
        _rows.append(r); probas[f"{fs} | {mn}"] = p
        print(f"  {fs:30s} {mn:7s} AUC={r['auc']:.3f} "
              f"[{r['auc_lo']:.3f}-{r['auc_hi']:.3f}]  "
              f"sens={r['sensitivity']:.3f} spec={r['specificity']:.3f}")

RQ1 = pd.DataFrame(_rows)[["features", "model", "auc", "auc_lo", "auc_hi",
                           "accuracy", "sensitivity", "specificity", "f1"]]
print(); display(RQ1.round(3))
RQ1.round(4).to_csv(Path(CFG.OUT_DIR) / "table5_rq1.csv", index=False)

# --- Figure: the gap between grey and red IS the ECG's contribution --------
plt.figure(figsize=(7, 6.2))
for fs, col in [("Demographics (age, sex, BMI)", "#8c8c8c"),
                ("ECG only", "#4C72B0"), ("ECG + demographics", "#C44E52")]:
    sub = RQ1[RQ1.features == fs]
    bm = sub.loc[sub.auc.idxmax(), "model"]
    p = probas[f"{fs} | {bm}"]
    fpr, tpr, _ = roc_curve(yv, p)
    plt.plot(fpr, tpr, lw=2.4, color=col,
             label=f"{fs} ({bm})\n   AUC = {roc_auc_score(yv, p):.3f}")
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Chance")
plt.xlabel("1 − Specificity"); plt.ylabel("Sensitivity")
plt.title(f"RQ1 — does the ECG add value beyond the chart?\n"
          f"AHI ≥ {AHI_THRESHOLD:g}, n = {len(yv)}")
plt.legend(loc="lower right", fontsize=8.5); plt.tight_layout()
plt.savefig(Path(CFG.OUT_DIR) / "fig10_rq1.png", dpi=150, bbox_inches="tight")
plt.show()

_d = RQ1[RQ1.features.str.startswith("Demo")].auc.max()
_e = RQ1[RQ1.features == "ECG only"].auc.max()
_c = RQ1[RQ1.features.str.startswith("ECG +")].auc.max()
print("=" * 66); print("RQ1 VERDICT"); print("=" * 66)
print(f"  Demographics alone : AUC {_d:.3f}")
print(f"  ECG alone          : AUC {_e:.3f}")
print(f"  Combined           : AUC {_c:.3f}")
if _e > _d + 0.05:
    print(f"\n  The ECG outperforms age/sex/BMI by {_e-_d:+.3f} AUC — it carries")
    print(f"  apnea information the patient's chart does not.")
elif _e > _d:
    print(f"\n  The ECG edges out demographics ({_e-_d:+.3f} AUC), but the bootstrap")
    print(f"  intervals overlap. Report as comparable, not superior.")
else:
    print(f"\n  Demographics match or beat the ECG here. State it plainly — at")
    print(f"  n={len(yv)} that is a real finding about this cohort's limits.")

---
# 6. Save the trained model

We retrain the best Stage 1 architecture on **all** available minutes and save the
weights, the Stage 2 classifier, and a configuration file the app reads.

The numbers we **report** stay the cross-validated and held-out figures from
sections 3 and 5. This refit exists only to produce a deployable artifact, and must
never be evaluated on data it has seen.

In [ ]:
# ---------------------------------------------------------------------------
# 4. Save the weights
# ---------------------------------------------------------------------------
# Retrains the best architecture on ALL minutes. Reported performance stays the
# cross-validated / held-out figures; this refit exists only to produce a
# deployable artifact and must never be evaluated on data it has seen.
print("\n" + "=" * 74)
print("4. SAVING WEIGHTS")
print("=" * 74)

# Prefer held-out test scores when they exist — a model chosen on the official
# test set is a more defensible basis than one chosen on cross-validation.
if globals().get("HAVE_TEST") and globals().get("RESULTS_TEST"):
    _board = dict(RESULTS_TEST); _basis = "held-out test set"
else:
    _board = {nm: pd.DataFrame(f).mean().to_dict() for nm, f in RESULTS_CV.items()}
    _basis = "cross-validation"

BEST_S1 = max(_board, key=lambda k: _board[k]["f1"])
# "RRI only" variants drop the EDR channel. The deployed model MUST use the
# same channel subset it was validated on — silently feeding it both channels
# would make predictions drift from the reported performance.
BEST_CH = [0] if "RRI only" in BEST_S1 else list(range(X_deep.shape[2]))

print(f"Selected on {_basis}:")
for nm, m in sorted(_board.items(), key=lambda kv: -kv[1]["f1"]):
    mark = "  <-- chosen" if nm == BEST_S1 else ""
    print(f"  {nm:26s} F1={m['f1']:.4f}  AUC={m.get('auc', float('nan')):.4f}{mark}")
print(f"\nChannels: {BEST_CH}  "
      f"({'RRI only' if BEST_CH == [0] else 'RRI + EDR'})")

print("\nRetraining on all minutes ...")
_tr, _va = split_train_val(np.arange(len(y)), groups, frac=0.15)
_mk = lambda idx: WindowDataset(X_deep, y, idx, EP_START, EP_END,
                                CFG.CONTEXT, BEST_CH)
_, _, _, DEPLOY_MODEL = train_fold(
    CNNBiLSTM(len(BEST_CH)), _mk(_tr), _mk(_va), _mk(_va),
    class_weights(y[_tr]), tag="deploy")
torch.save(DEPLOY_MODEL.state_dict(), ARTIFACTS / "cnn_bilstm.pt")

# --- Stage 2 classifier ----------------------------------------------------
best_row = RQ1.loc[RQ1.auc.idxmax()]
S2_COLS = FEATURE_SETS[best_row["features"]]
STAGE2_MODEL = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler()),
                         ("clf", MODELS[best_row["model"]]())]
                        ).fit(S2[S2_COLS].to_numpy(dtype=np.float64), yv)
pickle.dump(STAGE2_MODEL, open(ARTIFACTS / "stage2.pkl", "wb"))

# --- Screening threshold ---------------------------------------------------
# Screening favours sensitivity: missing an apneic patient costs far more than
# an unnecessary sleep study. Take the highest-specificity threshold that still
# reaches 90% sensitivity, computed from the LOSO probabilities rather than the
# refit model, which has already seen every patient.
_p = probas[f"{best_row['features']} | {best_row['model']}"]
_fpr, _tpr, _thr = roc_curve(yv, _p)
_ok = np.where(_tpr >= 0.90)[0]
SCREEN_THR = float(_thr[_ok[0]]) if len(_ok) else 0.5
SCREEN_SENS = float(_tpr[_ok[0]]) if len(_ok) else float("nan")
SCREEN_SPEC = float(1 - _fpr[_ok[0]]) if len(_ok) else float("nan")

DEPLOY_CFG = dict(
    fs=CFG.FS, segment_sec=CFG.SEGMENT_SEC, derived_fs=CFG.DERIVED_FS,
    derived_len=CFG.DERIVED_LEN, context=CFG.CONTEXT,
    ecg_band=list(CFG.ECG_BAND), butter_order=CFG.BUTTER_ORDER,
    channels=BEST_CH, stage1_model=BEST_S1, stage1_basis=_basis,
    stage1_f1=float(_board[BEST_S1]["f1"]),
    stage1_auc=float(_board[BEST_S1].get("auc", float("nan"))),
    stage2_features=S2_COLS, stage2_model=best_row["model"],
    stage2_auc=float(best_row.auc),
    stage2_auc_ci=[float(best_row.auc_lo), float(best_row.auc_hi)],
    ahi_threshold=AHI_THRESHOLD, screening_threshold=SCREEN_THR,
    screening_sensitivity=SCREEN_SENS, screening_specificity=SCREEN_SPEC,
    n_patients=int(len(yv)), n_minutes=int(len(y)),
    n_records=int(len(np.unique(groups))),
)
json.dump(DEPLOY_CFG, open(ARTIFACTS / "config.json", "w"), indent=2)

print(f"\nStage 2: {best_row['features']} / {best_row['model']}  "
      f"AUC {best_row.auc:.3f} [{best_row.auc_lo:.3f}-{best_row.auc_hi:.3f}]")
print(f"Screening threshold {SCREEN_THR:.3f} -> "
      f"sensitivity {SCREEN_SENS:.3f}, specificity {SCREEN_SPEC:.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# 5. Sample patients for the app
# ---------------------------------------------------------------------------
# Visitors at a poster will not have ECG files. Bundling a spread of severities
# means the demo can show both a positive and a negative result.
print("\n" + "=" * 74)
print("5. SAMPLE PATIENTS")
print("=" * 74)

_avail = DEMO.loc[[r for r in DEMO.index if r in RECORDS]].sort_values("AHI")
_picks = [_avail.index[-1], _avail.index[len(_avail) // 2], _avail.index[0]]
_index = {}
for rec in _picks:
    for ext in [".hea", ".dat"]:
        if os.path.exists(RECORDS[rec] + ext):
            shutil.copy(RECORDS[rec] + ext, SAMPLES / (rec + ext))
    d = DEMO.loc[rec]
    lab = ("Severe" if d.AHI >= 30 else "Moderate" if d.AHI >= 15
           else "Mild" if d.AHI >= 5 else "Normal")
    _index[f"{rec} — {lab} (AHI {d.AHI:.0f})"] = dict(
        record=rec, age=float(d.age), sex=d.sex,
        height_cm=float(d.height_cm), weight_kg=float(d.weight_kg),
        AHI=float(d.AHI))
json.dump(_index, open(SAMPLES / "index.json", "w"), indent=2)

for k, v in _index.items():
    print(f"  {k}   ({v['age']:.0f}y {v['sex']}, "
          f"BMI {v['weight_kg']/(v['height_cm']/100)**2:.1f})")


# ---------------------------------------------------------------------------
# 6. Done
# ---------------------------------------------------------------------------
print("\n" + "=" * 74)
print("FILES WRITTEN  (Output panel, right sidebar)")
print("=" * 74)
print("/kaggle/working/model/")
for p in sorted(ARTIFACTS.glob("*")):
    print(f"  {p.name:18s} {p.stat().st_size/1024:9.1f} KB")
print("/kaggle/working/samples/")
for p in sorted(SAMPLES.glob("*")):
    print(f"  {p.name:18s} {p.stat().st_size/1024:9.1f} KB")

_expect = ["cnn_bilstm.pt", "stage2.pkl", "config.json"]
_have = [f for f in _expect if (ARTIFACTS / f).exists()]
print(f"\n{len(_have)}/3 model files present.",
      "Ready for the app." if len(_have) == 3 else f"MISSING: {set(_expect)-set(_have)}")

In [ ]:
# ===========================================================================
# 6.c  Confirm what was written
# ===========================================================================
print("=" * 74)
print("FILES WRITTEN  (Output panel, right sidebar)")
print("=" * 74)
print("/kaggle/working/model/")
for p in sorted(ARTIFACTS.glob("*")):
    print(f"  {p.name:18s} {p.stat().st_size/1024:9.1f} KB")
print("/kaggle/working/samples/")
for p in sorted(SAMPLES.glob("*")):
    print(f"  {p.name:18s} {p.stat().st_size/1024:9.1f} KB")

_expect = ["cnn_bilstm.pt", "stage2.pkl", "config.json"]
_have = [f for f in _expect if (ARTIFACTS / f).exists()]
print(f"\n{len(_have)}/3 model files present.",
      "Ready for the app." if len(_have) == 3 else f"MISSING: {set(_expect)-set(_have)}")
print("\nDownload /kaggle/working/model/ and /kaggle/working/samples/ from the")
print("Output panel, then drop them into the Gradio app folder.")

---
# 7. Interpretation and limitations

## What the two comparisons mean

**Temporal context.** The 1D-CNN and CNN-BiLSTM share a byte-identical encoder and see identical
folds, so the gap between them isolates the sequence layer. Expect the BiLSTM to win, and expect the
gain to show up mostly as improved **sensitivity** — isolated ambiguous minutes get resolved by the
run they sit inside. If sensitivity rises while specificity barely moves, that is the mechanism.

**ECG-derived respiration.** If the RRI-only ablation matches the full model, EDR is redundant given
rhythm — itself a useful finding, since EDR is the more fragile of the two signals and dropping it
would simplify any deployment.

## Limitations, roughly in order of severity

**The cohort is small and old.** Seventy recordings from a study published in 2000, predominantly
male, from patients already referred for suspected sleep-disordered breathing. Base rates here are
far above a community population, and positive predictive value scales with prevalence — so these
numbers would degrade in an unselected screening population. This is the main barrier to reading
them as deployment estimates.

**Per-minute labels are not per-event labels.** A minute is scored apneic if apnea occurred during
it. That is coarser than the clinical AHI, which counts individual events with duration criteria. So
our "apnea index" correlates with AHI but is not AHI, and we should not report it as such.

**Scoring criteria have changed.** AASM definitions of hypopnea were revised after these recordings
were annotated. A model trained on 2000-era scoring may not match how a 2026 sleep laboratory would
label the same night.

**One night per patient.** Apnea severity varies substantially night to night, so a single recording
conflates model error with the patient's own variability.

**Single-lead ECG at 100 Hz.** R-peak timing resolves to 10 ms. Clinical HRV analysis typically uses
250 Hz or higher, so our short-term measures — RMSSD in particular — are noisier than published
norms.

**Stage 2 inherits Stage 1's errors,** and the Bland–Altman bias should be read alongside the
correlation. A high correlation with a systematic offset still means the index is wrong.

## What would strengthen this

External validation on a second cohort is the single most valuable addition — the UCD Sleep Apnea
Database (25 subjects, also Open Access, with Holter ECG) is a natural choice and would directly
address the "one small old dataset" objection. Beyond that, multi-night recordings would separate
model error from biological variability, and a modern wearable cohort would test whether the
approach survives the move from clinical electrodes to a consumer device.

## Positioning

Apnea-ECG is the reference benchmark for this task, and published methods reach very high per-minute
accuracy on it. We are not claiming to beat the state of the art. The contribution is the controlled
test of whether temporal context earns its complexity, the EDR ablation, and a screening-oriented
evaluation that reports the sensitivity–specificity trade-off at a clinically chosen operating point
rather than a single accuracy figure.